# 🧠 NeuroVerse — Motor Spiral & Wave Drawing Model Training
## Kaggle Notebook: Parkinson's Disease Detection via Drawing Analysis

### Clinical Background
**Spiral and wave drawing tests** are established clinical tools for detecting Parkinson's Disease (PD).
PD patients exhibit characteristic drawing impairments:
- **Tremor** — 4-6 Hz oscillations visible as wavy/jagged lines
- **Micrographia** — progressively smaller drawing amplitude
- **Bradykinesia** — slower drawing speed, uneven pressure
- **Rigidity** — loss of smooth curvature, angular deviations

### Datasets (Multi-Source — ~800+ images combined)

| # | Dataset | Source | Images | Types | Access |
|---|---------|--------|--------|-------|--------|
| 1 | **Parkinson's Drawings** (Zham et al., 2017) | Kaggle `kmader/parkinsons-drawings` | ~102 | Spiral + Wave | Direct Kaggle load |
| 2 | **Parkinson Drawing Spiral+Wave** | Kaggle `alihussainpk/parkinson-drawing-spiralwave` | ~200+ | Spiral + Wave | Direct Kaggle load |
| 3 | **Spiral Drawings PD** | Kaggle `vasanthakiranshahi/parkinson-spiral-drawings` | ~150+ | Spiral | Direct Kaggle load |
| 4 | **HandPD** (Pereira et al., 2016) | GitHub `biolab/handpd` | ~92 | Spiral + Meander | `git clone` (free) |
| 5 | **NewHandPD** (Pereira et al., 2016) | GitHub `biolab/handpd` | ~264 | Spiral + Meander | `git clone` (free) |

**Combined: ~800+ unique images → with 5x augmentation → ~4,000+ effective training samples**

### Architecture
- **EfficientNet-B0** (pretrained on ImageNet) with custom PD classification head
- **Dual-task**: Spiral classifier + Wave classifier → fused motor score
- **GradCAM** for XAI — highlights tremor regions in drawings
- **Data augmentation** (5x expansion with heavy transforms)

### Output
- `motor_spiral_model.pt` — PyTorch model for NeuroVerse backend
- UPDRS-aligned tremor severity scoring
- GradCAM visualization for doctor dashboard

## 1️⃣ Environment Setup & Multi-Dataset Loading

### Kaggle Datasets (add ALL of these in the sidebar → "Add Data")
1. `kmader/parkinsons-drawings` → `/kaggle/input/parkinsons-drawings/`
2. `alihussainpk/parkinson-drawing-spiralwave` → `/kaggle/input/parkinson-drawing-spiralwave/`
3. `vasanthakiranshahi/parkinson-spiral-drawings` → `/kaggle/input/parkinson-spiral-drawings/`

### Non-Kaggle Datasets (downloaded automatically via git clone)
4. **HandPD** + **NewHandPD** (Pereira et al., 2016) — from GitHub `biolab/handpd`

> **Note**: Add as many Kaggle datasets as available. The scanner handles missing datasets gracefully — if a dataset isn't added, it's skipped.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies & Load Dataset
# ============================================================
# 📌 BEFORE running: Upload HandPD_Meander.zip to Colab
#    Option A: Drag & drop into the Files panel (left sidebar) → loads at /content/
#    Option B: Upload to Google Drive → mount Drive below

# ══════════════════════════════════════════════════════════════
# ⚠️  IMPORTANT: If numpy was corrupted by a previous run,
#     go to Runtime → Restart session FIRST, then re-run this cell.
# ══════════════════════════════════════════════════════════════

# Step 1: Ensure numpy is healthy (DO NOT downgrade it!)
import importlib, sys
try:
    import numpy as np
    _ = np.array([1, 2, 3])  # Quick sanity check
    print(f"✅ numpy {np.__version__} is healthy")
except Exception:
    print("⚠️  numpy is corrupted — reinstalling...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy"])
    print("✅ numpy reinstalled — please RESTART RUNTIME and re-run this cell!")
    raise SystemExit("Restart runtime now!")

# Step 2: Install ML packages (safe — never touches numpy)
!pip install -q timm grad-cam lime scikit-learn
!pip install -q captum --no-deps

import os, glob, json, hashlib, warnings, random, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from PIL import Image

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, accuracy_score
)

import timm

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
# Reproducibility
# ═══════════════════════════════════════════════════════════
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# ═══════════════════════════════════════════════════════════
# Output directories
# ═══════════════════════════════════════════════════════════
OUTPUT_DIR = '/content/motor_meander_output'
MODEL_DIR  = os.path.join(OUTPUT_DIR, 'checkpoints')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'export'), exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 📦 Load Dataset — Choose ONE method below
# ═══════════════════════════════════════════════════════════
# METHOD A: File uploaded to Colab sidebar (appears at /content/)
# METHOD B: File on Google Drive (fastest for large zips)

USE_DRIVE = True   # ← Set False if you dragged zip into Colab sidebar

DATA_DIR = '/content/datasets'
os.makedirs(DATA_DIR, exist_ok=True)
MARKER = os.path.join(DATA_DIR, '.handpd_meander_extracted')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/NeuroVerse_Datasets/HandPD_Meander.zip'
else:
    ZIP_PATH = '/content/HandPD_Meander.zip'

assert os.path.exists(ZIP_PATH), (
    f"❌ {ZIP_PATH} not found!\n"
    f"   If USE_DRIVE=True: upload HandPD_Meander.zip to Google Drive → MyDrive/NeuroVerse_Datasets/\n"
    f"   If USE_DRIVE=False: drag & drop the zip into Colab's 📁 Files panel"
)

if os.path.exists(MARKER):
    print("✅ HandPD_Meander.zip already extracted — skipping")
else:
    print(f"📦 Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATA_DIR)
    open(MARKER, 'w').close()
    print("✅ Extraction complete!")

# ═══════════════════════════════════════════════════════════
# Auto-detect extracted HandPD folder
# ═══════════════════════════════════════════════════════════
def find_folder(base, candidates):
    for c in candidates:
        path = os.path.join(base, c)
        if os.path.isdir(path):
            return path
        for sub in (os.listdir(base) if os.path.isdir(base) else []):
            path2 = os.path.join(base, sub, c)
            if os.path.isdir(path2):
                return path2
    return None

HANDPD_ROOT = find_folder(DATA_DIR, ['HandPD_Augmented_Data', 'HandPD_Meander', 'meander'])
print(f"   HandPD root: {HANDPD_ROOT}")

# ═══════════════════════════════════════════════════════════
# LOAD HandPD — MEANDER ONLY (~12,638 images)
# ═══════════════════════════════════════════════════════════
all_records = []
valid_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}

healthy_names = {'healthymeander', 'aug_healthymeander'}
patient_names = {'patientmeander', 'aug_patientmeander'}

print("\n📂 Loading HandPD_Augmented_Data (meander only)...")

if HANDPD_ROOT and os.path.isdir(HANDPD_ROOT):
    aug_types = ['Geometric_Augmentation', 'Mixed_Augmentation', 'Photometric_Augmentation']

    for aug_type in aug_types:
        aug_count = 0
        for split_name in ['training_set', 'test_set']:
            meander_dir = os.path.join(HANDPD_ROOT, aug_type, split_name, 'meander')
            if not os.path.isdir(meander_dir):
                continue

            for class_folder in os.listdir(meander_dir):
                class_dir = os.path.join(meander_dir, class_folder)
                if not os.path.isdir(class_dir):
                    continue

                folder_lower = class_folder.lower()
                if folder_lower in healthy_names or folder_lower.startswith('healthy'):
                    class_name, label = 'healthy', 0
                elif folder_lower in patient_names or folder_lower.startswith('patient') or folder_lower.startswith('aug_patient'):
                    class_name, label = 'parkinson', 1
                else:
                    continue

                for img_file in os.listdir(class_dir):
                    if os.path.splitext(img_file)[1].lower() in valid_extensions:
                        all_records.append({
                            'path': os.path.join(class_dir, img_file),
                            'label': label,
                            'class_name': class_name,
                            'drawing_type': 'meander',
                            'dataset': f'HandPD_{aug_type.split("_")[0]}',
                            'split': 'train' if 'training' in split_name else 'test',
                        })
                        aug_count += 1

        print(f"   {aug_type.split('_')[0]:15s}: {aug_count} meander images")
else:
    raise FileNotFoundError(
        f"❌ HandPD dataset not found!\n"
        f"   Searched in: {DATA_DIR}\n"
        f"   Make sure your zip contains HandPD_Augmented_Data/ folder"
    )

# ═══════════════════════════════════════════════════════════
# Build & Clean DataFrame
# ═══════════════════════════════════════════════════════════
df_all = pd.DataFrame(all_records)
assert len(df_all) > 0, "❌ No meander images loaded! Check your dataset paths."

valid_mask = df_all['path'].apply(os.path.exists)
if (~valid_mask).sum() > 0:
    print(f"   ⚠️  Removing {(~valid_mask).sum()} missing files")
    df_all = df_all[valid_mask].reset_index(drop=True)

# Deduplicate by file hash
print("🔍 Deduplicating...")
df_all['hash'] = df_all['path'].apply(
    lambda p: hashlib.md5(open(p, 'rb').read(8192)).hexdigest() if os.path.exists(p) else None
)
before = len(df_all)
df_all = df_all.drop_duplicates(subset='hash', keep='first').drop(columns=['hash']).reset_index(drop=True)
if before - len(df_all) > 0:
    print(f"   Removed {before - len(df_all)} duplicates")

# ═══════════════════════════════════════════════════════════
# Dataset Statistics
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"📊 MEANDER DATASET SUMMARY")
print(f"{'='*70}")
print(f"   Total images: {len(df_all)}")
print(f"   Classes: {dict(df_all['class_name'].value_counts())}")
print(f"\n   By augmentation type:")
for ds, count in df_all['dataset'].value_counts().items():
    h = (df_all[df_all['dataset']==ds]['label']==0).sum()
    p = (df_all[df_all['dataset']==ds]['label']==1).sum()
    print(f"     {ds:25s} {count:5d} (H={h}, PD={p})")
print(f"\n   By split (original):")
for sp, count in df_all['split'].value_counts().items():
    print(f"     {sp:10s} {count:5d}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4ECDC4', '#FF6B6B']
df_all['class_name'].value_counts().plot.bar(ax=axes[0], color=colors)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count'); axes[0].tick_params(axis='x', rotation=0)

df_all.groupby(['dataset','class_name']).size().unstack(fill_value=0).plot.bar(ax=axes[1], color=colors, stacked=True)
axes[1].set_title('Per Augmentation Type', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45); axes[1].legend(title='Class')

df_all.groupby(['split','class_name']).size().unstack(fill_value=0).plot.bar(ax=axes[2], color=colors, stacked=True)
axes[2].set_title('Train vs Test (Original Split)', fontsize=13, fontweight='bold')
axes[2].tick_params(axis='x', rotation=0); axes[2].legend(title='Class')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'dataset_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

# Sample images
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Sample Meander Drawing Images', fontsize=16, fontweight='bold')
for i, (_, row) in enumerate(df_all.sample(10, random_state=SEED).iterrows()):
    r, c = i//5, i%5
    try:
        axes[r,c].imshow(Image.open(row['path']).convert('RGB'))
        axes[r,c].set_title(f"{row['dataset']}\n{row['class_name']}", fontsize=8,
                             color='green' if row['label']==0 else 'red')
    except: pass
    axes[r,c].axis('off')
plt.tight_layout(); plt.show()

print(f"\n✅ Dataset ready: {len(df_all)} meander images from {df_all['dataset'].nunique()} augmentation types")

## 2️⃣ Smart Multi-Dataset Scanner

The scanner auto-detects drawing type and class from **any folder structure**:
- Folder name contains `spiral`/`wave`/`meander` → drawing type
- Folder name contains `healthy`/`control`/`normal` → class = healthy
- Folder name contains `parkinson`/`patient`/`pd` → class = PD
- `train`/`test` → split assignment

### Supported structures (all handled automatically):
```
kmader:        spiral/training/healthy/*.png
alihussain:    spiral/healthy/*.png  OR  spiral_healthy/*.png
vasantha:      healthy/*.png  OR  parkinson/*.png
handpd:        HandPD/spiral/control/*.png, HandPD/spiral/patient/*.png
newhandpd:     NewHandPD/spiral/control/*.png, NewHandPD/spiral/patient/*.png
```

Datasets are **deduplicated by image hash** to avoid training on duplicate images.

In [ ]:
# ============================================================
# CELL 2: Smart Multi-Dataset Scanner (with deduplication)
# ============================================================
import hashlib

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif'}

# ═══════════════════════════════════════════════════════════
# Smart folder-name parser — works across ALL dataset formats
# ═══════════════════════════════════════════════════════════
def parse_path_metadata(full_path, dataset_root):
    """
    Parse drawing_type, split, and class from folder structure.
    Handles: kmader, alihussain, vasantha, HandPD, NewHandPD.
    """
    relative = full_path.replace(dataset_root, "").strip(os.sep).lower()
    parts = relative.replace("\\", "/").split("/")
    
    drawing_type = None
    split_name = None
    class_name = None
    
    for part in parts:
        # Drawing type detection
        if 'spiral' in part:
            drawing_type = 'spiral'
        elif 'wave' in part:
            drawing_type = 'wave'
        elif 'meander' in part:
            drawing_type = 'meander'  # HandPD uses meander (similar to wave)
        
        # Split detection
        if part in ('training', 'train'):
            split_name = 'train'
        elif part in ('testing', 'test'):
            split_name = 'test'
        
        # Class detection (broad matching)
        if part in ('healthy', 'health', 'control', 'controls', 'normal', 'h'):
            class_name = 'healthy'
        elif part in ('parkinson', 'parkinsons', 'patient', 'patients', 'pd', 'p'):
            class_name = 'parkinson'
    
    # Some datasets embed class in filename (e.g., "H_001.png" or "P_001.png")
    if class_name is None:
        fname = os.path.basename(full_path).lower()
        if fname.startswith(('h_', 'h0', 'healthy', 'control')):
            class_name = 'healthy'
        elif fname.startswith(('p_', 'p0', 'parkinson', 'patient', 'pd')):
            class_name = 'parkinson'
    
    # Treat meander as wave (both test motor fluidity)
    if drawing_type == 'meander':
        drawing_type = 'wave'
    
    # Default split to train if not specified
    if split_name is None:
        split_name = 'train'
    
    return drawing_type, split_name, class_name


def compute_image_hash(filepath, hash_size=8):
    """Fast perceptual hash for deduplication (uses file content hash)."""
    try:
        with open(filepath, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except:
        return None


# ═══════════════════════════════════════════════════════════
# Scan ALL active datasets
# ═══════════════════════════════════════════════════════════
all_images = []
seen_hashes = set()
duplicate_count = 0

for ds_key, ds_info in active_datasets.items():
    ds_path = ds_info["path"]
    ds_name = ds_info["name"]
    ds_count = 0
    ds_skipped = 0
    
    for root, dirs, files in os.walk(ds_path):
        for f in files:
            ext = Path(f).suffix.lower()
            if ext not in IMAGE_EXTS:
                continue
            
            full_path = os.path.join(root, f)
            
            # Deduplication via content hash
            img_hash = compute_image_hash(full_path)
            if img_hash and img_hash in seen_hashes:
                duplicate_count += 1
                ds_skipped += 1
                continue
            if img_hash:
                seen_hashes.add(img_hash)
            
            drawing_type, split_name, class_name = parse_path_metadata(full_path, ds_path)
            
            # Skip images where we can't determine the class
            if class_name is None:
                ds_skipped += 1
                continue
            
            # If drawing_type unknown, try to infer from dataset key
            if drawing_type is None:
                if 'spiral' in ds_key:
                    drawing_type = 'spiral'
                else:
                    drawing_type = 'spiral'  # default assumption
            
            all_images.append({
                'path': full_path,
                'drawing_type': drawing_type,
                'split': split_name,
                'class_name': class_name,
                'label': 0 if class_name == 'healthy' else 1,
                'filename': f,
                'dataset': ds_key,
                'dataset_name': ds_name,
            })
            ds_count += 1
    
    print(f"   📦 {ds_name}: {ds_count} valid images ({ds_skipped} skipped/duplicates)")

df_all = pd.DataFrame(all_images)

# ═══════════════════════════════════════════════════════════
# Comprehensive Summary
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("📊 COMBINED MULTI-DATASET SCAN RESULTS")
print("=" * 70)
print(f"\n   Total unique images: {len(df_all)}")
print(f"   Duplicates removed:  {duplicate_count}")
print(f"   Datasets loaded:     {df_all['dataset'].nunique()}")

print(f"\n   📋 By Dataset:")
for ds, count in df_all['dataset_name'].value_counts().items():
    print(f"     {ds}: {count}")

print(f"\n   📋 By Drawing Type:")
for dt, count in df_all['drawing_type'].value_counts().items():
    print(f"     {dt}: {count}")

print(f"\n   📋 By Class:")
for cls, count in df_all['class_name'].value_counts().items():
    print(f"     {cls}: {count}")

balance = df_all['class_name'].value_counts()
ratio = balance.min() / balance.max()
print(f"\n   ⚖️  Class balance: {ratio:.2f} (1.0 = perfect)")
if ratio < 0.7:
    print(f"   ⚠️  Imbalanced! WeightedRandomSampler will fix this.")

print(f"\n   📋 Detailed Breakdown:")
breakdown = df_all.groupby(['dataset', 'drawing_type', 'class_name']).size().reset_index(name='count')
for _, row in breakdown.iterrows():
    print(f"     {row['dataset']:20s} | {row['drawing_type']:8s} | {row['class_name']:10s} | {row['count']:4d}")

# Separate spiral and wave DataFrames (for later analysis)
df_spiral = df_all[df_all['drawing_type'] == 'spiral'].copy()
df_wave = df_all[df_all['drawing_type'] == 'wave'].copy()

print(f"\n   🌀 Spirals: {len(df_spiral)} | 🌊 Waves: {len(df_wave)}")
print(f"\n✅ Multi-dataset loading complete — {len(df_all)} unique images ready!")

## 3️⃣ Dataset Visualization — Healthy vs Parkinson's Drawings

Visual comparison of drawing quality differences between healthy controls and PD patients.

In [ ]:
# ============================================================
# CELL 3: Visualize Sample Drawings
# ============================================================

fig, axes = plt.subplots(2, 6, figsize=(22, 8))
fig.suptitle('Healthy vs Parkinson\'s — Spiral & Wave Drawings', fontsize=16, fontweight='bold')

# Row labels
axes[0, 0].set_ylabel('SPIRAL', fontsize=14, fontweight='bold', rotation=0, labelpad=60, va='center')
axes[1, 0].set_ylabel('WAVE', fontsize=14, fontweight='bold', rotation=0, labelpad=60, va='center')

for row_idx, drawing_type in enumerate(['spiral', 'wave']):
    df_type = df_all[df_all['drawing_type'] == drawing_type]
    
    # 3 healthy + 3 parkinson
    for col_idx, class_name in enumerate(['healthy', 'parkinson']):
        subset = df_type[df_type['class_name'] == class_name]
        samples = subset.sample(min(3, len(subset)), random_state=SEED)
        
        for i, (_, row) in enumerate(samples.iterrows()):
            ax_col = col_idx * 3 + i
            try:
                img = Image.open(row['path']).convert('RGB')
                axes[row_idx, ax_col].imshow(img)
            except Exception as e:
                axes[row_idx, ax_col].text(0.5, 0.5, 'Error', ha='center', va='center')
            
            color = 'green' if class_name == 'healthy' else 'red'
            axes[row_idx, ax_col].set_title(
                f"{'✅ Healthy' if class_name == 'healthy' else '🔴 PD'}",
                fontsize=10, color=color, fontweight='bold'
            )
            axes[row_idx, ax_col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'pd_drawing_samples.png'), dpi=150, bbox_inches='tight')
plt.show()
print("← Healthy Controls ───────── Parkinson's Disease →")

# ═══════════════════════════════════════════════════════════
# Image size distribution
# ═══════════════════════════════════════════════════════════
print(f"\n📐 Image Size Distribution:")
sizes = []
for _, row in df_all.sample(min(50, len(df_all))).iterrows():
    try:
        img = Image.open(row['path'])
        sizes.append(img.size)
    except:
        pass

widths = [s[0] for s in sizes]
heights = [s[1] for s in sizes]
print(f"   Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}")
print(f"   Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}")

## 4️⃣ Heavy Data Augmentation Strategy

### Why augmentation?
Even with ~800+ images from multiple sources, the dataset is **still relatively small** for deep learning.
Heavy augmentation prevents overfitting and simulates real-world variability:
- **Rotation ±30°** — paper orientation variation
- **Elastic deformation** — simulates natural pen stroke variation
- **Gaussian noise** — scanner/camera quality variation
- **Brightness/contrast jitter** — lighting conditions from different datasets
- **Perspective distortion** — camera/scanner angle differences
- **Random erasing** — small occlusions
- **Horizontal + Vertical flip** — drawings are symmetric (unlike clock drawings!)
- **Affine transformations** — scale/shift/shear

With 5x oversampling: **~800 images → ~4,000 effective training samples per epoch**

In [ ]:
# ============================================================
# CELL 4: Custom Dataset & Augmentation Pipeline
# ============================================================

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ═══════════════════════════════════════════════════════════
# HEAVY augmentation — 12 transforms for diverse training
# ═══════════════════════════════════════════════════════════
train_transforms = T.Compose([
    T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    T.RandomCrop(IMG_SIZE),
    T.RandomRotation(30),                                  # ±30° — generous rotation
    T.RandomHorizontalFlip(p=0.5),                         # OK for spirals/waves (symmetric)
    T.RandomVerticalFlip(p=0.3),                           # Also OK
    T.RandomPerspective(distortion_scale=0.15, p=0.4),     # Camera/scanner angle
    T.RandomAffine(
        degrees=0,
        translate=(0.08, 0.08),                            # Position shift
        scale=(0.85, 1.15),                                # Scale variation
        shear=(-10, 10),                                   # Shear deformation
    ),
    T.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.2,
        hue=0.05,
    ),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),      # Pen/scan blur
    T.RandomGrayscale(p=0.3),                              # Many originals are grayscale
    T.RandomInvert(p=0.1),                                 # Some scans have inverted colors
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.15)),           # Random occlusions
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ═══════════════════════════════════════════════════════════
# Custom Dataset — supports both spiral and wave drawings
# ═══════════════════════════════════════════════════════════
class ParkinsonsDrawingDataset(Dataset):
    """
    Parkinson's spiral/wave drawing dataset.
    
    Labels: 0 = Healthy, 1 = Parkinson's Disease
    """
    
    def __init__(self, dataframe, transform=None, oversample_factor=1):
        """
        Args:
            dataframe: DataFrame with 'path', 'label' columns
            transform: Image transforms
            oversample_factor: Repeat dataset N times per epoch (for small datasets)
        """
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform
        self.oversample_factor = oversample_factor
    
    def __len__(self):
        return len(self.data) * self.oversample_factor
    
    def __getitem__(self, idx):
        # Wrap around for oversampling
        real_idx = idx % len(self.data)
        row = self.data.iloc[real_idx]
        
        # Load image
        image = Image.open(row['path']).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, row['label']

CLASS_NAMES = {0: 'Healthy', 1: "Parkinson's"}
print(f"✅ Dataset and transforms defined")
print(f"   Classes: {CLASS_NAMES}")
print(f"   Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Train augmentation: 12 transforms (heavy for small dataset)")

## 🔬 DDPM — Diffusion-Based Synthetic Data Augmentation

Traditional augmentations (rotation, flip, color jitter) only *transform* existing images. A **Denoising Diffusion Probabilistic Model (DDPM)** learns the underlying distribution of meander/wave drawing patterns and generates **entirely new, realistic samples** — capturing subtle tremor patterns, pressure variations, and stroke morphologies that geometric transforms cannot synthesize.

### Architecture
- **Lightweight U-Net** (~8M params) — fits in Colab T4 VRAM
- **Class-conditional** — generates Healthy OR PD drawings on demand
- **Cosine β schedule** (500 timesteps) — better than linear (Nichol & Dhariwal, 2021)
- **64×64 generation → 224×224 bicubic upscale** — fast training, high-quality output

### Strategy
- Identify the **minority class** (Healthy or PD)
- Generate 120% of the imbalance gap as synthetic images
- Add to `df_all` BEFORE the train/val/test split

In [ ]:
# ============================================================
# CELL 4B: DDPM Architecture — Lightweight Conditional Diffusion
# ============================================================

DDPM_IMG_SIZE = 64          # Generate at 64×64, upscale later
DDPM_TIMESTEPS = 500        # Diffusion steps
DDPM_EPOCHS = 100           # Training epochs (~15 min on T4)
DDPM_BATCH_SIZE = 64
DDPM_LR = 2e-4
USE_DIFFUSION = True        # Set False to skip DDPM

# ═══════════════════════════════════════════════════════════
# Cosine Noise Schedule (Improved DDPM, Nichol & Dhariwal 2021)
# ═══════════════════════════════════════════════════════════
def cosine_beta_schedule(timesteps, s=0.008):
    """Cosine schedule as proposed in 'Improved DDPM'."""
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)

# Precompute schedule tensors
betas = cosine_beta_schedule(DDPM_TIMESTEPS)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

def extract(a, t, x_shape):
    """Extract values from schedule tensor at timestep t."""
    batch_size = t.shape[0]
    out = a.gather(-1, t.cpu())
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(t.device)

# ═══════════════════════════════════════════════════════════
# Forward Diffusion: q(x_t | x_0)
# ═══════════════════════════════════════════════════════════
def q_sample(x_start, t, noise=None):
    """Add noise to image at timestep t."""
    if noise is None:
        noise = torch.randn_like(x_start)
    sqrt_alpha = extract(sqrt_alphas_cumprod, t, x_start.shape)
    sqrt_one_minus_alpha = extract(sqrt_one_minus_alphas_cumprod, t, x_start.shape)
    return sqrt_alpha * x_start + sqrt_one_minus_alpha * noise

# ═══════════════════════════════════════════════════════════
# Sinusoidal Time Embedding
# ═══════════════════════════════════════════════════════════
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

# ═══════════════════════════════════════════════════════════
# Residual Block with Time + Class Conditioning
# ═══════════════════════════════════════════════════════════
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn1 = nn.GroupNorm(8, out_ch)
        self.bn2 = nn.GroupNorm(8, out_ch)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch))
        self.class_emb = nn.Embedding(num_classes, out_ch)
        self.residual = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb, class_label):
        h = self.bn1(F.silu(self.conv1(x)))
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = h + self.class_emb(class_label)[:, :, None, None]
        h = self.bn2(F.silu(self.conv2(h)))
        return h + self.residual(x)

# ═══════════════════════════════════════════════════════════
# Lightweight U-Net for 64×64 Diffusion
# ═══════════════════════════════════════════════════════════
class LightweightUNet(nn.Module):
    """
    Compact U-Net for class-conditional DDPM.
    ~8M params — fits in Colab T4 GPU (~1.5GB VRAM).
    """
    def __init__(self, in_channels=3, base_ch=64, time_dim=256, num_classes=2):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim), nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        # Encoder
        self.enc1 = ResBlock(in_channels, base_ch, time_dim, num_classes)
        self.enc2 = ResBlock(base_ch, base_ch * 2, time_dim, num_classes)
        self.enc3 = ResBlock(base_ch * 2, base_ch * 4, time_dim, num_classes)
        self.down1 = nn.Conv2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.down3 = nn.Conv2d(base_ch * 4, base_ch * 4, 4, stride=2, padding=1)
        # Bottleneck
        self.bottleneck = ResBlock(base_ch * 4, base_ch * 4, time_dim, num_classes)
        # Decoder
        self.up3 = nn.ConvTranspose2d(base_ch * 4, base_ch * 4, 4, stride=2, padding=1)
        self.dec3 = ResBlock(base_ch * 8, base_ch * 2, time_dim, num_classes)
        self.up2 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.dec2 = ResBlock(base_ch * 4, base_ch, time_dim, num_classes)
        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.dec1 = ResBlock(base_ch * 2, base_ch, time_dim, num_classes)
        self.out_conv = nn.Conv2d(base_ch, in_channels, 1)
    
    def forward(self, x, timestep, class_label):
        t = self.time_mlp(timestep)
        e1 = self.enc1(x, t, class_label)
        e2 = self.enc2(self.down1(e1), t, class_label)
        e3 = self.enc3(self.down2(e2), t, class_label)
        b = self.bottleneck(self.down3(e3), t, class_label)
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1), t, class_label)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1), t, class_label)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1), t, class_label)
        return self.out_conv(d1)

# ═══════════════════════════════════════════════════════════
# DDPM Sampling (Reverse Diffusion)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def p_sample(model, x, t, t_index, class_label):
    betas_t = extract(betas, t, x.shape)
    sqrt_one_minus_alpha_t = extract(sqrt_one_minus_alphas_cumprod, t, x.shape)
    sqrt_recip_alpha_t = extract(sqrt_recip_alphas, t, x.shape)
    predicted_noise = model(x, t, class_label)
    model_mean = sqrt_recip_alpha_t * (x - betas_t * predicted_noise / sqrt_one_minus_alpha_t)
    if t_index == 0:
        return model_mean
    posterior_var_t = extract(posterior_variance, t, x.shape)
    return model_mean + torch.sqrt(posterior_var_t) * torch.randn_like(x)

@torch.no_grad()
def p_sample_loop(model, shape, class_label, device):
    img = torch.randn(shape, device=device)
    for i in reversed(range(DDPM_TIMESTEPS)):
        t = torch.full((shape[0],), i, device=device, dtype=torch.long)
        img = p_sample(model, img, t, i, class_label)
    return img

@torch.no_grad()
def generate_samples(model, num_samples, target_class, device, batch_size=32):
    model.eval()
    all_images = []
    for i in range(0, num_samples, batch_size):
        bs = min(batch_size, num_samples - i)
        class_labels = torch.full((bs,), target_class, device=device, dtype=torch.long)
        images = p_sample_loop(model, (bs, 3, DDPM_IMG_SIZE, DDPM_IMG_SIZE), class_labels, device)
        images = torch.clamp(images, -1, 1) * 0.5 + 0.5
        all_images.append(images.cpu())
        print(f"   Generated {min(i + bs, num_samples)}/{num_samples}", end='\r')
    return torch.cat(all_images, dim=0)

NUM_CLASSES = 2
print("✅ DDPM architecture defined")
params = sum(p.numel() for p in LightweightUNet().parameters())
print(f"   U-Net parameters: {params:,} ({params/1e6:.1f}M)")
print(f"   Diffusion steps: {DDPM_TIMESTEPS}")
print(f"   Generation resolution: {DDPM_IMG_SIZE}×{DDPM_IMG_SIZE}")
print(f"   Noise schedule: cosine (Nichol & Dhariwal, 2021)")

In [ ]:
# ============================================================
# CELL 4C: Train DDPM & Generate Synthetic Meander Data
# ============================================================
# 1. Trains conditional DDPM on ALL meander images (~15-20 min on T4)
# 2. Generates synthetic minority-class images
# 3. Saves to disk & adds to df_all (BEFORE train/val/test split)
# ============================================================

if USE_DIFFUSION:
    print("=" * 70)
    print("🔬 DDPM SYNTHETIC DATA GENERATION — MEANDER DRAWINGS")
    print("=" * 70)
    
    # ─── Prepare 64×64 dataset for DDPM training ───
    ddpm_transforms = T.Compose([
        T.Resize((DDPM_IMG_SIZE, DDPM_IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # Scale to [-1, 1]
    ])
    
    class DDPMDataset(Dataset):
        def __init__(self, dataframe, transform):
            self.data = dataframe.reset_index(drop=True)
            self.transform = transform
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            row = self.data.iloc[idx]
            img = Image.open(row['path']).convert('RGB')
            return self.transform(img), row['label']
    
    ddpm_dataset = DDPMDataset(df_all, ddpm_transforms)
    ddpm_loader = DataLoader(
        ddpm_dataset, batch_size=DDPM_BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True
    )
    
    # ─── Class distribution analysis ───
    class_dist = df_all['label'].value_counts().sort_index()
    minority_class = class_dist.idxmin()
    majority_class = class_dist.idxmax()
    imbalance_gap = class_dist[majority_class] - class_dist[minority_class]
    
    print(f"\n📊 Current class distribution:")
    print(f"   Healthy (0): {class_dist.get(0, 0)} images")
    print(f"   PD (1):      {class_dist.get(1, 0)} images")
    print(f"   Minority class: {CLASS_NAMES[minority_class]} (label={minority_class})")
    print(f"   Imbalance gap: {imbalance_gap} images")
    
    n_synthetic = int(imbalance_gap * 1.2) if imbalance_gap > 100 else 500
    print(f"   Will generate: {n_synthetic} synthetic {CLASS_NAMES[minority_class]} meander images")
    
    # ─── Initialize model ───
    ddpm_model = LightweightUNet(
        in_channels=3, base_ch=64, time_dim=256, num_classes=NUM_CLASSES
    ).to(device)
    
    ddpm_optimizer = torch.optim.AdamW(ddpm_model.parameters(), lr=DDPM_LR, weight_decay=1e-4)
    ddpm_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ddpm_optimizer, T_max=DDPM_EPOCHS)
    
    # ─── Training Loop ───
    print(f"\n🏋️ Training DDPM on meander drawings for {DDPM_EPOCHS} epochs...")
    print(f"   Batches/epoch: {len(ddpm_loader)}")
    
    ddpm_losses = []
    best_ddpm_loss = float('inf')
    
    for epoch in range(DDPM_EPOCHS):
        ddpm_model.train()
        epoch_loss = 0.0
        
        for batch_imgs, batch_labels in ddpm_loader:
            batch_imgs = batch_imgs.to(device)
            batch_labels = batch_labels.to(device).long()
            
            t = torch.randint(0, DDPM_TIMESTEPS, (batch_imgs.shape[0],), device=device).long()
            noise = torch.randn_like(batch_imgs)
            x_noisy = q_sample(batch_imgs, t, noise)
            predicted_noise = ddpm_model(x_noisy, t, batch_labels)
            loss = F.mse_loss(predicted_noise, noise)
            
            ddpm_optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ddpm_model.parameters(), 1.0)
            ddpm_optimizer.step()
            epoch_loss += loss.item()
        
        ddpm_scheduler.step()
        avg_loss = epoch_loss / len(ddpm_loader)
        ddpm_losses.append(avg_loss)
        
        if avg_loss < best_ddpm_loss:
            best_ddpm_loss = avg_loss
            torch.save(ddpm_model.state_dict(), os.path.join(MODEL_DIR, 'ddpm_meander_best.pt'))
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1:3d}/{DDPM_EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_ddpm_loss:.4f} | LR: {ddpm_scheduler.get_last_lr()[0]:.2e}")
    
    print(f"\n✅ DDPM training complete! Best loss: {best_ddpm_loss:.4f}")
    
    # ─── Plot training loss ───
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    ax.plot(ddpm_losses, color='purple', linewidth=1.5)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.set_title('DDPM Training Loss — Meander Drawings')
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    
    # ─── Load best model & generate synthetic images ───
    ddpm_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'ddpm_meander_best.pt'), weights_only=True))
    
    print(f"\n🎨 Generating {n_synthetic} synthetic {CLASS_NAMES[minority_class]} images...")
    synthetic_images = generate_samples(ddpm_model, n_synthetic, minority_class, device, batch_size=32)
    
    # ─── Save synthetic images to disk ───
    synthetic_dir = os.path.join(DATA_DIR, 'synthetic_meander')
    os.makedirs(synthetic_dir, exist_ok=True)
    
    synthetic_records = []
    upscale = T.Resize((224, 224), interpolation=T.InterpolationMode.BICUBIC, antialias=True)
    
    for i, img_tensor in enumerate(synthetic_images):
        img_up = upscale(img_tensor)
        img_pil = T.ToPILImage()(img_up)
        save_path = os.path.join(synthetic_dir, f'synth_{CLASS_NAMES[minority_class].lower()}_{i:04d}.png')
        img_pil.save(save_path, quality=95)
        synthetic_records.append({
            'path': save_path,
            'label': minority_class,
            'class_name': 'healthy' if minority_class == 0 else 'parkinson',
            'drawing_type': 'meander',
            'dataset': 'DDPM_Synthetic',
            'split': 'train',
        })
    
    df_synthetic = pd.DataFrame(synthetic_records)
    
    # ─── Inject into df_all BEFORE splitting ───
    df_all_original = len(df_all)
    df_all = pd.concat([df_all, df_synthetic], ignore_index=True)
    
    print(f"\n📊 Dataset after DDPM augmentation:")
    print(f"   Original: {df_all_original} images")
    print(f"   Synthetic: +{len(df_synthetic)} {CLASS_NAMES[minority_class]} images")
    print(f"   Total:    {len(df_all)} images")
    
    new_dist = df_all['label'].value_counts().sort_index()
    ratio = new_dist.min() / new_dist.max()
    print(f"\n   New distribution:")
    print(f"   Healthy (0): {new_dist.get(0, 0)} images")
    print(f"   PD (1):      {new_dist.get(1, 0)} images")
    print(f"   Balance ratio: {ratio:.2f}")
    
    # ─── Visualize: Real vs Synthetic ───
    fig, axes = plt.subplots(2, 8, figsize=(20, 6))
    fig.suptitle(f'Real vs Synthetic {CLASS_NAMES[minority_class]} Meander Drawings', fontsize=14, fontweight='bold')
    
    real_minority = df_all[(df_all['label'] == minority_class) & (df_all['dataset'] != 'DDPM_Synthetic')]
    for i in range(min(8, len(real_minority))):
        row = real_minority.iloc[i]
        try:
            img = Image.open(row['path']).convert('RGB').resize((128, 128))
            axes[0, i].imshow(img)
        except:
            pass
        axes[0, i].set_title('Real', fontsize=9, color='blue')
        axes[0, i].axis('off')
    
    for i in range(min(8, len(synthetic_images))):
        img_np = synthetic_images[i].permute(1, 2, 0).numpy()
        img_np = np.clip(img_np, 0, 1)
        axes[1, i].imshow(img_np)
        axes[1, i].set_title('Synthetic', fontsize=9, color='purple')
        axes[1, i].axis('off')
    
    axes[0, 0].set_ylabel('Real', fontsize=12, fontweight='bold', rotation=0, labelpad=40)
    axes[1, 0].set_ylabel('Synthetic', fontsize=12, fontweight='bold', rotation=0, labelpad=40)
    plt.tight_layout(); plt.show()
    
    # Clean up GPU memory
    del ddpm_model, ddpm_optimizer, ddpm_scheduler, synthetic_images
    torch.cuda.empty_cache()
    print(f"\n✅ DDPM complete — {len(df_synthetic)} synthetic images added to dataset")

else:
    print("⏭️  DDPM skipped (USE_DIFFUSION = False)")
    print(f"   Dataset size: {len(df_all)} images (real only)")

In [ ]:
# ============================================================
# CELL 5: Train/Val/Test Split & DataLoaders
# ============================================================

# ═══════════════════════════════════════════════════════════
# Strategy: Train on ALL drawing types together (wave/meander)
# The model learns general PD motor impairment patterns
# DDPM synthetic images are already in df_all at this point
# ═══════════════════════════════════════════════════════════

# For datasets with pre-defined splits, respect them — but only if
# the test set is large enough for reliable evaluation (>= 10% of total).
df_presplit_test = df_all[df_all['split'] == 'test'].copy()
df_presplit_train = df_all[df_all['split'] == 'train'].copy()

min_test_size = max(100, int(len(df_all) * 0.10))  # At least 100 or 10%

if len(df_presplit_test) >= min_test_size:
    # Pre-defined test set is large enough
    df_train_full = df_presplit_train.copy()
    df_test_orig = df_presplit_test.copy()
    print(f"   Using pre-defined test split: {len(df_test_orig)} images")
else:
    # Pre-defined test set too small ({len(df_presplit_test)} < {min_test_size})
    # → Re-split everything for proper evaluation
    print(f"   Pre-defined test set too small ({len(df_presplit_test)} < {min_test_size})")
    print(f"   Re-splitting all {len(df_all)} images into 70/15/15...")
    df_train_full, df_test_orig = train_test_split(
        df_all, test_size=0.15, random_state=SEED,
        stratify=df_all['class_name']
    )

# Create validation set from training data
try:
    stratify_col = df_train_full[['class_name', 'drawing_type']].apply(tuple, axis=1)
    min_group = stratify_col.value_counts().min()
    if min_group < 2:
        raise ValueError("Group too small for combined stratification")
    df_train, df_val = train_test_split(
        df_train_full, test_size=0.15, random_state=SEED,
        stratify=stratify_col
    )
except (ValueError, KeyError):
    df_train, df_val = train_test_split(
        df_train_full, test_size=0.15, random_state=SEED,
        stratify=df_train_full['class_name']
    )

print("=" * 70)
print("📊 SPLIT SUMMARY (meander dataset)")
print("=" * 70)
print(f"   Train: {len(df_train)} images")
print(f"   Val:   {len(df_val)} images")
print(f"   Test:  {len(df_test_orig)} images")
print(f"   Total: {len(df_train) + len(df_val) + len(df_test_orig)} images")

for split_name, df_split in [("Train", df_train), ("Val", df_val), ("Test", df_test_orig)]:
    print(f"\n   {split_name} breakdown:")
    for _, row in df_split.groupby(['drawing_type', 'class_name']).size().reset_index(name='n').iterrows():
        print(f"     {row['drawing_type']:20s} | {row['class_name']:10s} | {row['n']}")
    print(f"     Sources: {', '.join(df_split['dataset'].unique())}")

# ═══════════════════════════════════════════════════════════
# Weighted Sampler for balanced training
# ═══════════════════════════════════════════════════════════
class_counts = df_train['label'].value_counts().sort_index()
raw_weights = 1.0 / class_counts.values
# Normalize so weights sum to num_classes (for readable display)
norm_weights = raw_weights / raw_weights.sum() * len(class_counts)
sample_weights = df_train['label'].map(dict(zip(class_counts.index, raw_weights))).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print(f"\n   Class counts: Healthy={class_counts.get(0, 0)}, PD={class_counts.get(1, 0)}")
print(f"   Sampling weights (normalized): Healthy={norm_weights[0]:.3f}, PD={norm_weights[1]:.3f}")
print(f"   → Healthy sampled {norm_weights[0]/norm_weights[1]:.1f}x more often to balance")

# ═══════════════════════════════════════════════════════════
# Oversampling + batch size — adaptive based on dataset size
# ═══════════════════════════════════════════════════════════
n_train = len(df_train)
if n_train < 200:
    OVERSAMPLE = 10
    BATCH_SIZE = 16
elif n_train < 500:
    OVERSAMPLE = 5
    BATCH_SIZE = 32
elif n_train < 2000:
    OVERSAMPLE = 3
    BATCH_SIZE = 32
else:
    OVERSAMPLE = 1      # Large dataset — no need to repeat
    BATCH_SIZE = 64

train_dataset = ParkinsonsDrawingDataset(df_train, transform=train_transforms, oversample_factor=OVERSAMPLE)
val_dataset = ParkinsonsDrawingDataset(df_val, transform=val_transforms)
test_dataset = ParkinsonsDrawingDataset(df_test_orig, transform=val_transforms)

# WeightedRandomSampler for oversampled dataset
oversample_weights = np.tile(sample_weights, OVERSAMPLE)
oversample_sampler = WeightedRandomSampler(
    weights=oversample_weights,
    num_samples=len(oversample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=oversample_sampler,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"\n   Dataset size: {n_train} → Oversampling: {OVERSAMPLE}x → {len(train_dataset)} effective samples")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches/epoch: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

# ═══════════════════════════════════════════════════════════
# Preview augmented samples
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Augmented Training Samples (same image → different augmentations)', fontsize=14, fontweight='bold')

for i in range(12):
    row, col = i // 6, i % 6
    img, label = train_dataset[i]
    img_show = img.permute(1, 2, 0).numpy()
    img_show = img_show * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_show = np.clip(img_show, 0, 1)
    axes[row, col].imshow(img_show)
    axes[row, col].set_title(CLASS_NAMES[label], fontsize=9,
                              color='green' if label == 0 else 'red')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()
print(f"✅ DataLoaders ready with {OVERSAMPLE}x oversampling!")

## 5️⃣ Model Architecture — MotorNet (EfficientNet-B0 + PD Head)

### Why EfficientNet-B0 for small datasets?
- **ImageNet pretraining** transfers low-level features (edges, textures, curves) directly useful for tremor detection
- **5.3M parameters** but only ~800K trainable after freezing — reduces overfitting risk
- **Compound scaling** — optimal depth/width/resolution balance
- **GradCAM-ready** — conv_head layer gives clinically interpretable saliency maps

### Architecture:
```
EfficientNet-B0 (freeze blocks 0-4, fine-tune blocks 5-6)
    ↓ 1280-d feature vector
Dropout(0.5) → Linear(1280, 256) → ReLU → BatchNorm → Dropout(0.3)
    ↓
Linear(256, 64) → ReLU → BatchNorm
    ↓
Linear(64, 2) ← Healthy vs PD
    ↓ (parallel branch)
Linear(2, 16) → ReLU → Linear(16, 1) → Sigmoid ← PD risk [0,1]
```

### Higher dropout for small dataset:
- `Dropout(0.5)` at the top layer — aggressive regularization
- Smaller hidden dimensions (256→64 instead of 512→128)

In [ ]:
# ============================================================
# CELL 6: Model Architecture — MotorNet
# ============================================================

class MotorNet(nn.Module):
    """
    Parkinson's Disease motor drawing classifier based on EfficientNet-B0.
    
    Trained on spiral + wave drawings from Zham et al. (2017).
    Outputs:
    - class_logits: [Healthy, PD] binary classification
    - pd_risk: Continuous PD risk score [0, 1]
    
    GradCAM compatible — target_layer = self.backbone.conv_head
    """
    
    def __init__(self, num_classes=2, pretrained=True, dropout=0.5):
        super().__init__()
        
        # ── EfficientNet-B0 backbone ──
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=pretrained,
            num_classes=0,       # Remove classifier head
            global_pool='avg'    # → 1280-d feature vector
        )
        
        # Freeze bottom layers (blocks 0-4)
        # Fine-tune blocks 5, 6, conv_head, bn2
        for name, param in self.backbone.named_parameters():
            if any(x in name for x in ['blocks.5', 'blocks.6', 'conv_head', 'bn2']):
                param.requires_grad = True
            else:
                param.requires_grad = False
        
        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        frozen = sum(p.numel() for p in self.backbone.parameters() if not p.requires_grad)
        
        # ── Classification head (smaller for tiny dataset) ──
        feature_dim = self.backbone.num_features  # 1280
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),                    # 0.5 — aggressive for small data
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.6),              # 0.3
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(64),
            nn.Linear(64, num_classes),
        )
        
        # ── PD Risk head (regression) ──
        self.risk_head = nn.Sequential(
            nn.Linear(num_classes, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
        
        head_params = sum(p.numel() for p in self.classifier.parameters()) + \
                      sum(p.numel() for p in self.risk_head.parameters())
        
        print(f"🏗️  MotorNet Architecture:")
        print(f"   Backbone: EfficientNet-B0 ({frozen:,} frozen + {trainable:,} trainable)")
        print(f"   Head: {head_params:,} parameters")
        print(f"   Total trainable: {trainable + head_params:,}")
        print(f"   Dropout: {dropout} (aggressive for small dataset)")
        print(f"   Output: {num_classes} classes + PD risk score")
    
    def forward(self, x):
        features = self.backbone(x)          # [B, 1280]
        logits = self.classifier(features)   # [B, 2]
        
        # PD risk from class probabilities
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
        risk = self.risk_head(probs)         # [B, 1]
        
        return logits, risk.squeeze(-1)
    
    def get_features(self, x):
        """Get backbone features for GradCAM."""
        return self.backbone(x)


# ═══════════════════════════════════════════════════════════
# Instantiate model
# ═══════════════════════════════════════════════════════════
NUM_CLASSES = 2
model = MotorNet(num_classes=NUM_CLASSES, pretrained=True, dropout=0.5).to(device)

# Verify with dummy input
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
logits, risk = model(dummy)
print(f"\n✅ Forward pass test:")
print(f"   Input:  {dummy.shape}")
print(f"   Logits: {logits.shape} → {CLASS_NAMES}")
print(f"   Risk:   {risk.shape} (PD risk scores)")
print(f"   Risk values: {risk.detach().cpu().numpy()}")

## 6️⃣ Training Configuration

### Strategy for Small Dataset:
- **Label Smoothing (0.15)** — stronger than CDT (0.1) to prevent overconfident predictions
- **Weight Decay (5e-4)** — stronger L2 regularization
- **Cosine Annealing with Warm Restarts** — escape local minima
- **Early Stopping (patience=15)** — generous patience due to small/noisy validation set
- **Gradient clipping** — stability for small batches

In [ ]:
# ============================================================
# CELL 7: Training Loop with Early Stopping
# ============================================================

# ═══════════════════════════════════════════════════════════
# Hyperparameters (tuned for small dataset)
# ═══════════════════════════════════════════════════════════
NUM_EPOCHS = 60
LEARNING_RATE = 5e-4        # Lower than CDT — careful with small data
WEIGHT_DECAY = 5e-4         # Stronger L2 regularization
LABEL_SMOOTHING = 0.15      # Stronger smoothing for overfit prevention
PATIENCE = 15               # Generous patience (small val set = noisy metrics)

# ═══════════════════════════════════════════════════════════
# Loss, Optimizer, Scheduler
# ═══════════════════════════════════════════════════════════
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# Differential learning rates
backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
head_params = list(model.classifier.parameters()) + list(model.risk_head.parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': LEARNING_RATE / 10},   # Backbone: very slow
    {'params': head_params, 'lr': LEARNING_RATE},              # Head: normal
], weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7
)

# Mixed precision
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

# ═══════════════════════════════════════════════════════════
# Training functions
# ═══════════════════════════════════════════════════════════
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0
    all_preds, all_labels = [], []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                logits, risk = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, risk = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        logits, risk = model(images)
        loss = criterion(logits, labels)
        
        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs)
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)


# ═══════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("🚀 STARTING TRAINING — MotorNet (Spiral + Wave)")
print("=" * 60)
print(f"   Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"   Oversample: {OVERSAMPLE}x | Label Smoothing: {LABEL_SMOOTHING}")
print(f"   Early stopping patience: {PATIENCE}")
print()

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'train_f1': [], 'val_f1': [],
    'lr': []
}

best_val_f1 = 0
patience_counter = 0
best_epoch = 0

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device
    )
    
    # Validate
    val_loss, val_acc, val_f1, _, _, _ = validate(
        model, val_loader, criterion, device
    )
    
    # Scheduler step
    scheduler.step()
    current_lr = optimizer.param_groups[1]['lr']
    
    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)
    
    # Print
    print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS} │ "
          f"Train: loss={train_loss:.4f} acc={train_acc:.3f} f1={train_f1:.3f} │ "
          f"Val: loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f} │ "
          f"LR: {current_lr:.2e}", end="")
    
    # Save best
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        patience_counter = 0
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
            'num_classes': NUM_CLASSES,
            'img_size': IMG_SIZE,
            'class_names': CLASS_NAMES,
        }, os.path.join(MODEL_DIR, 'motor_best.pt'))
        
        print(f" ★ BEST", end="")
    else:
        patience_counter += 1
    
    print()
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n⏹️  Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
        break

print(f"\n{'='*60}")
print(f"✅ TRAINING COMPLETE!")
print(f"   Best epoch: {best_epoch} with val F1 = {best_val_f1:.4f}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 8: Training Curves
# ============================================================

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Validation', linewidth=2)
axes[1].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['train_f1'], label='Train', linewidth=2)
axes[2].plot(history['val_f1'], label='Validation', linewidth=2)
axes[2].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[2].set_title('Weighted F1 Score', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Learning Rate
axes[3].plot(history['lr'], linewidth=2, color='purple')
axes[3].set_title('Learning Rate', fontsize=14, fontweight='bold')
axes[3].set_xlabel('Epoch')
axes[3].set_yscale('log')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'motor_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7️⃣ Test Set Evaluation

Load the best checkpoint and evaluate on the held-out test set.
Since this is a binary task, we get:
- **Accuracy**, **F1**, **Sensitivity** (true positive rate for PD), **Specificity** (true negative rate)
- **ROC-AUC** — area under the receiver operating characteristic curve
- **Confusion matrix** — shows false positives vs false negatives

In [ ]:
# ============================================================
# CELL 9: Test Set Evaluation
# ============================================================

# Load best model
checkpoint = torch.load(os.path.join(MODEL_DIR, 'motor_best.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']} (val F1={checkpoint['val_f1']:.4f})")

# Evaluate on test set
test_loss, test_acc, test_f1, test_preds, test_labels, test_probs = validate(
    model, test_loader, criterion, device
)

print(f"\n{'='*60}")
print(f"📊 TEST SET RESULTS")
print(f"{'='*60}")
print(f"   Accuracy:      {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"   F1 (weighted): {test_f1:.4f}")
print(f"   Loss:          {test_loss:.4f}")

# Per-class report
class_names_list = ['Healthy', "Parkinson's"]
print(f"\n📋 Classification Report:")
print(classification_report(test_labels, test_preds, target_names=class_names_list, digits=3))

# ROC-AUC
try:
    auc = roc_auc_score(test_labels, test_probs[:, 1])
    print(f"   ROC-AUC: {auc:.4f}")
except Exception as e:
    print(f"   ROC-AUC: Could not compute ({e})")

# Sensitivity & Specificity
cm = confusion_matrix(test_labels, test_preds)
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    print(f"\n   Clinical Metrics:")
    print(f"   Sensitivity (PD detection): {sensitivity:.3f} ({sensitivity*100:.1f}%)")
    print(f"   Specificity (Healthy correct): {specificity:.3f} ({specificity*100:.1f}%)")
    print(f"   PPV (Positive Predictive Value): {ppv:.3f}")
    print(f"   NPV (Negative Predictive Value): {npv:.3f}")

# ═══════════════════════════════════════════════════════════
# Confusion Matrix
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[0],
            annot_kws={"size": 18})
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[1],
            annot_kws={"size": 18})
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'motor_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════
# Separate evaluation: Spiral-only vs Wave-only
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"📊 SEPARATE EVALUATION — Spiral vs Wave")
print(f"{'='*60}")

for drawing_type in ['spiral', 'wave']:
    df_type_test = df_test_orig[df_test_orig['drawing_type'] == drawing_type]
    if len(df_type_test) == 0:
        continue
    
    type_dataset = ParkinsonsDrawingDataset(df_type_test, transform=val_transforms)
    type_loader = DataLoader(type_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    _, type_acc, type_f1, type_preds, type_labels, type_probs = validate(
        model, type_loader, criterion, device
    )
    
    type_auc = roc_auc_score(type_labels, type_probs[:, 1]) if len(np.unique(type_labels)) > 1 else 0
    
    print(f"\n   {drawing_type.upper()}:")
    print(f"     Accuracy: {type_acc:.3f} | F1: {type_f1:.3f} | AUC: {type_auc:.3f}")
    print(f"     ({len(df_type_test)} test images)")

## 8️⃣ GradCAM — Explainable AI for Motor Drawings

### What GradCAM reveals for PD drawings:
- **Tremor hotspots** — jagged/wavy regions where hand shook
- **Micrographia** — areas where spiral gets progressively tighter
- **Irregularity zones** — breaks in smooth curvature (rigidity)
- **Pen pressure variation** — darker/lighter regions from uneven pressure

This is critical for NeuroVerse's XAI module — doctors can see **exactly where** the model detected PD signs in the patient's drawing.

In [ ]:
# ============================================================
# CELL 10: GradCAM Implementation
# ============================================================

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

class MotorModelWrapper(nn.Module):
    """Wrapper that returns only logits (GradCAM needs single output)."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logits, _ = self.model(x)
        return logits

wrapped_model = MotorModelWrapper(model)

# Target layer — last convolutional layer of EfficientNet-B0
target_layer = model.backbone.conv_head

# Initialize GradCAM
cam = GradCAM(model=wrapped_model, target_layers=[target_layer])

# ═══════════════════════════════════════════════════════════
# Generate GradCAM for a single image
# ═══════════════════════════════════════════════════════════
def generate_gradcam(image_path, true_label=None):
    """Generate GradCAM for a drawing image."""
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    # Prediction
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        pred_class = logits.argmax(dim=1).item()
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        risk_val = risk.item()
    
    # GradCAM for predicted class
    targets = [ClassifierOutputTarget(pred_class)]
    grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
    
    # Overlay
    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_resized).astype(np.float32) / 255.0
    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    
    return {
        'original': img_np,
        'gradcam': cam_image,
        'heatmap': grayscale_cam,
        'pred_class': pred_class,
        'true_label': true_label,
        'probs': probs,
        'risk': risk_val
    }


# ═══════════════════════════════════════════════════════════
# Visualize: Healthy vs PD — Spiral + Wave GradCAMs
# ═══════════════════════════════════════════════════════════
print("🔍 Generating GradCAM visualizations...")

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('GradCAM: Healthy vs PD — What the Model Sees\n'
             '(Red = High Attention for PD detection | Blue = Low Attention)',
             fontsize=16, fontweight='bold')

columns = [
    ('spiral', 'healthy', 'Spiral — Healthy'),
    ('spiral', 'parkinson', 'Spiral — PD'),
    ('wave', 'healthy', 'Wave — Healthy'),
    ('wave', 'parkinson', 'Wave — PD'),
]

for col_idx, (drawing_type, class_name, title) in enumerate(columns):
    subset = df_test_orig[
        (df_test_orig['drawing_type'] == drawing_type) &
        (df_test_orig['class_name'] == class_name)
    ]
    
    if len(subset) == 0:
        subset = df_all[
            (df_all['drawing_type'] == drawing_type) &
            (df_all['class_name'] == class_name)
        ]
    
    if len(subset) == 0:
        for row in range(3):
            axes[row, col_idx].axis('off')
        continue
    
    sample = subset.sample(1, random_state=SEED + col_idx).iloc[0]
    result = generate_gradcam(sample['path'], true_label=sample['label'])
    
    # Row 0: Original
    axes[0, col_idx].imshow(result['original'])
    axes[0, col_idx].set_title(title, fontsize=11, fontweight='bold',
                                color='green' if class_name == 'healthy' else 'red')
    axes[0, col_idx].axis('off')
    
    # Row 1: GradCAM overlay
    axes[1, col_idx].imshow(result['gradcam'])
    pred_name = CLASS_NAMES[result['pred_class']]
    correct = result['pred_class'] == sample['label']
    axes[1, col_idx].set_title(
        f"Pred: {pred_name} ({'✓' if correct else '✗'})\n"
        f"PD Risk: {result['risk']:.2f}",
        fontsize=9, color='green' if correct else 'red'
    )
    axes[1, col_idx].axis('off')
    
    # Row 2: Heatmap only
    axes[2, col_idx].imshow(result['heatmap'], cmap='jet')
    conf = result['probs'][result['pred_class']]
    axes[2, col_idx].set_title(f"Confidence: {conf:.1%}", fontsize=9)
    axes[2, col_idx].axis('off')

# Row labels
for i, label in enumerate(['Original', 'GradCAM', 'Heatmap']):
    axes[i, 0].set_ylabel(label, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'motor_gradcam_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ GradCAM visualizations saved!")

In [ ]:
# ============================================================
# CELL 11: GradCAM — All Test Samples + Misclassified Analysis
# ============================================================

# Find correct and misclassified examples
correct_mask = test_preds == test_labels
correct_indices = np.where(correct_mask)[0]
wrong_indices = np.where(~correct_mask)[0]

print(f"   Correct: {len(correct_indices)} | Misclassified: {len(wrong_indices)}")
print(f"   Test Accuracy: {len(correct_indices)/(len(correct_indices)+len(wrong_indices)):.1%}")

if len(wrong_indices) > 0:
    n_show = min(6, len(wrong_indices))
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    fig.suptitle('GradCAM on MISCLASSIFIED Samples\n(What confused the model?)',
                 fontsize=14, fontweight='bold')
    
    if n_show == 1:
        axes = axes.reshape(2, 1)
    
    for i, idx in enumerate(wrong_indices[:n_show]):
        row = df_test_orig.iloc[idx]
        result = generate_gradcam(row['path'], true_label=int(test_labels[idx]))
        
        # Original
        axes[0, i].imshow(result['original'])
        axes[0, i].set_title(
            f"True: {CLASS_NAMES[int(test_labels[idx])]}\n"
            f"Pred: {CLASS_NAMES[int(test_preds[idx])]}",
            fontsize=9, color='red', fontweight='bold'
        )
        axes[0, i].axis('off')
        
        # GradCAM
        axes[1, i].imshow(result['gradcam'])
        axes[1, i].set_title(
            f"{row['drawing_type'].upper()}\n"
            f"Risk: {result['risk']:.2f} | Conf: {result['probs'][result['pred_class']]:.1%}",
            fontsize=8
        )
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'motor_misclassified_gradcam.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("🎉 No misclassifications on test set!")

## 8.5️⃣ Multi-XAI — GradCAM++, LIME, Integrated Gradients & Occlusion Sensitivity

Beyond GradCAM, we apply **4 additional XAI methods** for comprehensive explainability:

| Method | What It Shows | Why It Matters |
|--------|--------------|----------------|
| **GradCAM++** | Weighted gradient activations (improved) | Better localization for multiple tremor regions |
| **LIME** | Superpixel importance via perturbation | Model-agnostic, shows which image parts matter |
| **Integrated Gradients** | Pixel-level attribution (Captum) | Axiomatic method — theoretically grounded |
| **Occlusion Sensitivity** | Systematic patch masking | Shows what happens when parts are hidden |

All 4 methods should converge on the same regions (tremor lines, irregular edges) — this **cross-validates** our GradCAM findings.

In [ ]:
# ============================================================
# CELL 11.5A: Multi-XAI — GradCAM++, LIME, Integrated Gradients, Occlusion
# ============================================================
# Provides 4 additional explainability methods beyond GradCAM

import torch.nn.functional as F
from pytorch_grad_cam import GradCAMPlusPlus
from captum.attr import IntegratedGradients, Occlusion
from lime import lime_image
from skimage.segmentation import mark_boundaries

# ═══════════════════════════════════════════════════════════
# 1. GradCAM++ (Improved GradCAM — better multi-region localization)
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("1️⃣  GradCAM++ — Improved Gradient-weighted Activation Maps")
print("=" * 60)

cam_pp = GradCAMPlusPlus(
    model=GradCAMWrapper(model),
    target_layers=[model.backbone.conv_head]
)

# Pick 3 PD + 3 Healthy test samples
n_samples = 3
pd_samples = df_test_orig[df_test_orig['label'] == 1].sample(n=n_samples, random_state=SEED)
hc_samples = df_test_orig[df_test_orig['label'] == 0].sample(n=n_samples, random_state=SEED)
xai_samples = pd.concat([pd_samples, hc_samples]).reset_index(drop=True)

fig, axes = plt.subplots(3, 6, figsize=(24, 12))
fig.suptitle('GradCAM++ — Enhanced Tremor Region Localization', fontsize=16, fontweight='bold')

for i, (_, row) in enumerate(xai_samples.iterrows()):
    img_pil = Image.open(row['path']).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_resized).astype(np.float32) / 255.0

    # Get prediction
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = logits.argmax(dim=1).item()

    true_label = int(row['label'])
    targets = [ClassifierOutputTarget(pred_idx)]
    grayscale_cam = cam_pp(input_tensor=img_tensor, targets=targets)[0]
    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

    # Row 0: Original
    axes[0, i].imshow(img_pil)
    color = 'green' if pred_idx == true_label else 'red'
    axes[0, i].set_title(f"True: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]:.1%})",
                         fontsize=9, color=color, fontweight='bold')
    axes[0, i].axis('off')

    # Row 1: GradCAM++ overlay
    axes[1, i].imshow(cam_image)
    axes[1, i].set_title(f"GradCAM++ Overlay", fontsize=9)
    axes[1, i].axis('off')

    # Row 2: Heatmap only
    axes[2, i].imshow(grayscale_cam, cmap='jet')
    axes[2, i].set_title(f"Risk: {risk.item() if hasattr(risk, 'item') else probs[1]:.2f}", fontsize=9)
    axes[2, i].axis('off')

for j, label in enumerate(['Original', 'GradCAM++', 'Heatmap']):
    axes[j, 0].set_ylabel(label, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'meander_gradcampp.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ GradCAM++ visualizations saved!")

del cam_pp
torch.cuda.empty_cache() if device.type == 'cuda' else None

In [ ]:
# ============================================================
# CELL 11.5B: LIME — Local Interpretable Model-agnostic Explanations
# ============================================================
# Perturbs superpixels and trains a local linear model to explain each prediction.
# Model-agnostic: treats the network as a black box.

print("=" * 60)
print("2️⃣  LIME — Superpixel Importance via Perturbation")
print("=" * 60)

explainer = lime_image.LimeImageExplainer()

def lime_predict_fn(images):
    """LIME prediction function — takes numpy arrays, returns probabilities."""
    batch = torch.stack([
        val_transforms(Image.fromarray(img.astype(np.uint8))) for img in images
    ]).to(device)
    model.eval()
    with torch.no_grad():
        logits, _ = model(batch)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs

# Pick 4 samples (2 PD, 2 Healthy)
lime_pd = df_test_orig[df_test_orig['label'] == 1].sample(n=2, random_state=SEED)
lime_hc = df_test_orig[df_test_orig['label'] == 0].sample(n=2, random_state=SEED)
lime_samples = pd.concat([lime_pd, lime_hc]).reset_index(drop=True)

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle("LIME — Which Superpixels Drive the Prediction?", fontsize=16, fontweight='bold')

for i, (_, row) in enumerate(lime_samples.iterrows()):
    img_pil = Image.open(row['path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_pil)
    true_label = int(row['label'])

    # Get prediction
    pred_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = model(pred_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = logits.argmax(dim=1).item()

    # LIME explanation (1000 perturbations)
    explanation = explainer.explain_instance(
        img_np, lime_predict_fn,
        top_labels=NUM_CLASSES, hide_color=0, num_samples=1000,
        random_seed=SEED
    )

    # Row 0: Original
    axes[0, i].imshow(img_np)
    color = 'green' if pred_idx == true_label else 'red'
    axes[0, i].set_title(f"True: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]:.1%})",
                         fontsize=9, color=color, fontweight='bold')
    axes[0, i].axis('off')

    # Row 1: Positive evidence (green = supports prediction)
    temp, mask = explanation.get_image_and_mask(
        pred_idx, positive_only=True, num_features=10, hide_rest=False
    )
    axes[1, i].imshow(mark_boundaries(temp / 255.0, mask))
    axes[1, i].set_title("Positive Evidence\n(supports prediction)", fontsize=9, color='green')
    axes[1, i].axis('off')

    # Row 2: Pro vs Con (green = supports, red = contradicts)
    temp2, mask2 = explanation.get_image_and_mask(
        pred_idx, positive_only=False, num_features=10, hide_rest=False
    )
    axes[2, i].imshow(mark_boundaries(temp2 / 255.0, mask2))
    axes[2, i].set_title("Pro (green) vs Con (red)", fontsize=9)
    axes[2, i].axis('off')

for j, label in enumerate(['Original', 'Positive LIME', 'Pro vs Con']):
    axes[j, 0].set_ylabel(label, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'meander_lime.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ LIME visualizations saved!")

In [ ]:
# ============================================================
# CELL 11.5C: Integrated Gradients — Pixel-Level Attribution (Captum)
# ============================================================
# Axiomatic attribution method (Sundararajan et al., 2017).
# Accumulates gradients along a path from a baseline (black image) to input.
# Theoretically grounded — satisfies Sensitivity and Implementation Invariance axioms.

print("=" * 60)
print("3️⃣  Integrated Gradients — Pixel-Level Attribution")
print("=" * 60)

class IGWrapper(nn.Module):
    """Wrapper for Captum — returns logits only."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        logits, _ = self.model(x)
        return logits

ig_model = IGWrapper(model).eval()
ig = IntegratedGradients(ig_model)

# Use same 4 samples as LIME for comparison
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle("Integrated Gradients — Pixel-Level Importance", fontsize=16, fontweight='bold')

for i, (_, row) in enumerate(lime_samples.iterrows()):
    img_pil = Image.open(row['path']).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    img_tensor.requires_grad = True
    true_label = int(row['label'])

    # Get prediction
    with torch.no_grad():
        logits, _ = model(img_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = logits.argmax(dim=1).item()

    # Compute IG attribution (50 steps along the path)
    baseline = torch.zeros_like(img_tensor).to(device)
    attributions = ig.attribute(
        img_tensor, baselines=baseline,
        target=pred_idx, n_steps=50,
        internal_batch_size=16
    )

    # Convert to displayable heatmap
    attr_np = attributions.squeeze(0).cpu().detach().numpy()
    attr_magnitude = np.abs(attr_np).sum(axis=0)  # Sum across RGB channels
    attr_magnitude = attr_magnitude / (attr_magnitude.max() + 1e-8)  # Normalize to [0, 1]

    # Original image for overlay
    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_resized).astype(np.float32) / 255.0

    # Row 0: Original
    axes[0, i].imshow(img_np)
    color = 'green' if pred_idx == true_label else 'red'
    axes[0, i].set_title(f"True: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]:.1%})",
                         fontsize=9, color=color, fontweight='bold')
    axes[0, i].axis('off')

    # Row 1: IG heatmap
    axes[1, i].imshow(attr_magnitude, cmap='hot')
    axes[1, i].set_title("Integrated Gradients\n(bright = important)", fontsize=9)
    axes[1, i].axis('off')

    # Row 2: Overlay
    overlay = img_np.copy()
    # Create red channel overlay where attribution is high
    alpha = 0.6
    heatmap_rgb = plt.cm.jet(attr_magnitude)[:, :, :3]
    blended = (1 - alpha) * img_np + alpha * heatmap_rgb.astype(np.float32)
    blended = np.clip(blended, 0, 1)
    axes[2, i].imshow(blended)
    axes[2, i].set_title("IG Overlay on Drawing", fontsize=9)
    axes[2, i].axis('off')

for j, label in enumerate(['Original', 'IG Attribution', 'IG Overlay']):
    axes[j, 0].set_ylabel(label, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'meander_integrated_gradients.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Integrated Gradients visualizations saved!")

del ig_model, ig
torch.cuda.empty_cache() if device.type == 'cuda' else None

In [ ]:
# ============================================================
# CELL 11.5D: Occlusion Sensitivity — Systematic Patch Masking
# ============================================================
# Slides a gray patch across the image and measures how much the
# prediction changes when each region is hidden.
# If hiding a region DROPS confidence → that region was important.

print("=" * 60)
print("4️⃣  Occlusion Sensitivity — What Happens When We Hide Regions?")
print("=" * 60)

occ_model = IGWrapper(model).eval()
occlusion = Occlusion(occ_model)

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle("Occlusion Sensitivity — Prediction Drop When Regions Hidden",
             fontsize=16, fontweight='bold')

for i, (_, row) in enumerate(lime_samples.iterrows()):
    img_pil = Image.open(row['path']).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    true_label = int(row['label'])

    with torch.no_grad():
        logits, _ = model(img_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = logits.argmax(dim=1).item()

    # Occlusion with 16×16 sliding window, stride 8
    attr_occ = occlusion.attribute(
        img_tensor, target=pred_idx,
        strides=(3, 8, 8),           # (C, H, W) stride
        sliding_window_shapes=(3, 16, 16),  # Occlude 16×16 patches
        baselines=0                   # Replace with black (0)
    )

    attr_np = attr_occ.squeeze(0).cpu().detach().numpy()
    attr_magnitude = np.abs(attr_np).sum(axis=0)
    attr_magnitude = attr_magnitude / (attr_magnitude.max() + 1e-8)

    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_resized).astype(np.float32) / 255.0

    # Row 0: Original
    axes[0, i].imshow(img_np)
    color = 'green' if pred_idx == true_label else 'red'
    axes[0, i].set_title(f"True: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]:.1%})",
                         fontsize=9, color=color, fontweight='bold')
    axes[0, i].axis('off')

    # Row 1: Occlusion heatmap
    axes[1, i].imshow(attr_magnitude, cmap='hot')
    axes[1, i].set_title("Occlusion Sensitivity\n(bright = hiding drops confidence)", fontsize=9)
    axes[1, i].axis('off')

    # Row 2: Overlay
    heatmap_rgb = plt.cm.jet(attr_magnitude)[:, :, :3]
    blended = 0.4 * img_np + 0.6 * heatmap_rgb.astype(np.float32)
    blended = np.clip(blended, 0, 1)
    axes[2, i].imshow(blended)
    axes[2, i].set_title("Occlusion Overlay", fontsize=9)
    axes[2, i].axis('off')

for j, label in enumerate(['Original', 'Occlusion Map', 'Overlay']):
    axes[j, 0].set_ylabel(label, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'meander_occlusion_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Occlusion Sensitivity visualizations saved!")

del occ_model, occlusion
torch.cuda.empty_cache() if device.type == 'cuda' else None

In [ ]:
# ============================================================
# CELL 11.5E: Multi-XAI Comparison Summary — All 5 Methods Side-by-Side
# ============================================================

print("=" * 60)
print("5️⃣  Multi-XAI Comparison — All Methods on Same Samples")
print("=" * 60)

# Pick 1 PD and 1 Healthy for clean comparison
comp_pd = df_test_orig[df_test_orig['label'] == 1].sample(n=1, random_state=42).iloc[0]
comp_hc = df_test_orig[df_test_orig['label'] == 0].sample(n=1, random_state=42).iloc[0]

for sample_row, sample_name in [(comp_pd, "Parkinson's"), (comp_hc, "Healthy")]:
    img_pil = Image.open(sample_row['path']).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_resized).astype(np.float32) / 255.0
    true_label = int(sample_row['label'])

    with torch.no_grad():
        logits, risk = model(img_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = logits.argmax(dim=1).item()

    fig, axes = plt.subplots(1, 6, figsize=(30, 5))
    fig.suptitle(f"Multi-XAI Comparison — True: {sample_name} | Pred: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]:.1%})",
                 fontsize=14, fontweight='bold',
                 color='green' if pred_idx == true_label else 'red')

    # 1. Original
    axes[0].imshow(img_np)
    axes[0].set_title("Original", fontsize=11, fontweight='bold')
    axes[0].axis('off')

    # 2. GradCAM
    cam_gc = GradCAM(model=GradCAMWrapper(model), target_layers=[model.backbone.conv_head])
    gc_map = cam_gc(input_tensor=img_tensor, targets=[ClassifierOutputTarget(pred_idx)])[0]
    axes[1].imshow(show_cam_on_image(img_np, gc_map, use_rgb=True))
    axes[1].set_title("GradCAM", fontsize=11, fontweight='bold')
    axes[1].axis('off')
    del cam_gc

    # 3. GradCAM++
    cam_pp2 = GradCAMPlusPlus(model=GradCAMWrapper(model), target_layers=[model.backbone.conv_head])
    gcpp_map = cam_pp2(input_tensor=img_tensor, targets=[ClassifierOutputTarget(pred_idx)])[0]
    axes[2].imshow(show_cam_on_image(img_np, gcpp_map, use_rgb=True))
    axes[2].set_title("GradCAM++", fontsize=11, fontweight='bold')
    axes[2].axis('off')
    del cam_pp2

    # 4. LIME
    lime_exp = explainer.explain_instance(
        np.array(img_resized), lime_predict_fn,
        top_labels=NUM_CLASSES, hide_color=0, num_samples=500, random_seed=SEED
    )
    temp_lime, mask_lime = lime_exp.get_image_and_mask(
        pred_idx, positive_only=True, num_features=8, hide_rest=False
    )
    axes[3].imshow(mark_boundaries(temp_lime / 255.0, mask_lime))
    axes[3].set_title("LIME", fontsize=11, fontweight='bold')
    axes[3].axis('off')

    # 5. Integrated Gradients
    ig_model2 = IGWrapper(model).eval()
    ig2 = IntegratedGradients(ig_model2)
    img_t2 = img_tensor.clone().detach().requires_grad_(True)
    ig_attr = ig2.attribute(img_t2, baselines=torch.zeros_like(img_t2).to(device),
                            target=pred_idx, n_steps=50, internal_batch_size=16)
    ig_np = np.abs(ig_attr.squeeze(0).cpu().detach().numpy()).sum(axis=0)
    ig_np = ig_np / (ig_np.max() + 1e-8)
    ig_overlay = 0.4 * img_np + 0.6 * plt.cm.jet(ig_np)[:, :, :3]
    axes[4].imshow(np.clip(ig_overlay, 0, 1))
    axes[4].set_title("Integrated\nGradients", fontsize=11, fontweight='bold')
    axes[4].axis('off')
    del ig_model2, ig2

    # 6. Occlusion
    occ_model2 = IGWrapper(model).eval()
    occ2 = Occlusion(occ_model2)
    occ_attr = occ2.attribute(img_tensor, target=pred_idx,
                              strides=(3, 8, 8), sliding_window_shapes=(3, 16, 16), baselines=0)
    occ_np = np.abs(occ_attr.squeeze(0).cpu().detach().numpy()).sum(axis=0)
    occ_np = occ_np / (occ_np.max() + 1e-8)
    occ_overlay = 0.4 * img_np + 0.6 * plt.cm.jet(occ_np)[:, :, :3]
    axes[5].imshow(np.clip(occ_overlay, 0, 1))
    axes[5].set_title("Occlusion\nSensitivity", fontsize=11, fontweight='bold')
    axes[5].axis('off')
    del occ_model2, occ2

    plt.tight_layout()
    tag = 'pd' if true_label == 1 else 'healthy'
    plt.savefig(os.path.join(MODEL_DIR, f'meander_multi_xai_{tag}.png'), dpi=150, bbox_inches='tight')
    plt.show()

    torch.cuda.empty_cache() if device.type == 'cuda' else None

print("\n" + "=" * 60)
print("✅ MULTI-XAI ANALYSIS COMPLETE")
print("=" * 60)
print("   Methods applied:")
print("   1. GradCAM        — gradient-weighted class activation maps")
print("   2. GradCAM++      — improved multi-region localization")
print("   3. LIME           — model-agnostic superpixel importance")
print("   4. Integrated Gradients — axiomatic pixel-level attribution")
print("   5. Occlusion Sensitivity — systematic patch masking")
print()
print("   All 5 methods should highlight similar regions:")
print("   • PD drawings → tremor lines, irregular edges, overshoots")
print("   • Healthy drawings → smooth continuous lines, even spacing")
print()
print("   Saved to:", MODEL_DIR)
print("   • meander_gradcampp.png")
print("   • meander_lime.png")
print("   • meander_integrated_gradients.png")
print("   • meander_occlusion_sensitivity.png")
print("   • meander_multi_xai_pd.png")
print("   • meander_multi_xai_healthy.png")

## 9️⃣ Full Fine-tuning (Unfreeze All Layers)

After the head is well-trained, unfreeze ALL backbone layers for a few more epochs at very low LR.
This typically adds **2-5% accuracy** on small datasets — the backbone layers adapt to the specific drawing domain.

In [ ]:
# ============================================================
# CELL 12: Full Fine-tuning (Optional)
# ============================================================

FULL_FINETUNE = True  # Set False to skip

if FULL_FINETUNE:
    print("🔓 Unfreezing ALL backbone layers for full fine-tuning...")
    
    for param in model.parameters():
        param.requires_grad = True
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable parameters: {total_params:,}")
    
    FT_LR = 5e-6       # Very conservative for small dataset
    FT_EPOCHS = 20
    FT_PATIENCE = 10
    
    ft_optimizer = optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)
    ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS, eta_min=1e-8)
    
    print(f"   LR: {FT_LR} | Epochs: {FT_EPOCHS} | Patience: {FT_PATIENCE}")
    print()
    
    ft_patience_counter = 0
    
    for epoch in range(FT_EPOCHS):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, ft_optimizer, scaler, device
        )
        val_loss, val_acc, val_f1, _, _, _ = validate(model, val_loader, criterion, device)
        ft_scheduler.step()
        
        print(f"  FT Epoch {epoch+1:2d}/{FT_EPOCHS} │ "
              f"Train: acc={train_acc:.3f} f1={train_f1:.3f} │ "
              f"Val: acc={val_acc:.3f} f1={val_f1:.3f}", end="")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            ft_patience_counter = 0
            torch.save({
                'epoch': NUM_EPOCHS + epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': ft_optimizer.state_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc,
                'num_classes': NUM_CLASSES,
                'img_size': IMG_SIZE,
                'class_names': CLASS_NAMES,
                'full_finetuned': True,
            }, os.path.join(MODEL_DIR, 'motor_best.pt'))
            print(f" ★ NEW BEST (f1={val_f1:.4f})")
        else:
            ft_patience_counter += 1
            print()
        
        if ft_patience_counter >= FT_PATIENCE:
            print(f"\n⏹️  Fine-tuning stopped (no improvement for {FT_PATIENCE} epochs)")
            break
    
    print(f"\n✅ Fine-tuning complete! Best F1: {best_val_f1:.4f}")
    
    # Reload best
    checkpoint = torch.load(os.path.join(MODEL_DIR, 'motor_best.pt'), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    print("⏭️  Skipping full fine-tuning")

## 🔟 Model Export for NeuroVerse Backend

Export the trained model in formats for deployment:
1. **`motor_spiral_model.pt`** — Full PyTorch model for backend (`app/ml/models/`)
2. **`motor_spiral_model_mobile.pt`** — TorchScript for mobile
3. **`motor_config.json`** — Class mapping + UPDRS thresholds for `fusion_service.py`

### How it maps to UPDRS (MDS-UPDRS 3.15-3.18):
| Prediction | UPDRS Tremor | Clinical Meaning |
|-----------|-------------|-----------------|
| Healthy (conf > 0.90) | 0 | Normal |
| Healthy (conf 0.70-0.90) | 1 | Slight tremor possible |
| PD (conf 0.50-0.70) | 2 | Mild tremor |
| PD (conf 0.70-0.85) | 3 | Moderate tremor |
| PD (conf > 0.85) | 4 | Severe tremor |

In [ ]:
# ============================================================
# CELL 13: Model Export
# ============================================================

EXPORT_DIR = os.path.join(OUTPUT_DIR, "export")
os.makedirs(EXPORT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 1. Full PyTorch model (for backend)
# ═══════════════════════════════════════════════════════════
export_path = os.path.join(EXPORT_DIR, "motor_spiral_model.pt")

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'architecture': 'efficientnet_b0',
        'num_classes': NUM_CLASSES,
        'img_size': IMG_SIZE,
        'dropout': 0.5,
    },
    'class_mapping': {
        'class_names': CLASS_NAMES,
        'label_to_class': {0: 'healthy', 1: 'parkinson'},
    },
    'normalization': {
        'mean': IMAGENET_MEAN,
        'std': IMAGENET_STD,
    },
    'metrics': {
        'test_accuracy': float(test_acc),
        'test_f1': float(test_f1),
        'best_val_f1': float(best_val_f1),
    },
    'clinical_thresholds': {
        # Maps model confidence → UPDRS tremor score (3.15-3.18)
        'updrs_mapping': {
            'healthy_high_conf':   {'min_conf': 0.90, 'updrs_tremor': 0, 'label': 'Normal'},
            'healthy_moderate':    {'min_conf': 0.70, 'updrs_tremor': 1, 'label': 'Slight'},
            'pd_low_conf':         {'min_conf': 0.50, 'updrs_tremor': 2, 'label': 'Mild tremor'},
            'pd_moderate_conf':    {'min_conf': 0.70, 'updrs_tremor': 3, 'label': 'Moderate tremor'},
            'pd_high_conf':        {'min_conf': 0.85, 'updrs_tremor': 4, 'label': 'Severe tremor'},
        },
        # Risk mapping for fusion_service.py
        'pd_risk_by_prediction': {
            'healthy': {'pd_risk': 0.05, 'ad_risk': 0.02},
            'parkinson': {'pd_risk': 0.65, 'ad_risk': 0.05},
        },
        # Tremor frequency range (PD-characteristic)
        'pd_tremor_frequency_hz': [4.0, 6.0],
        # Tapping threshold from fusion_service.py
        'tapping_pd_threshold': 3.0,
    },
    'dataset_info': {
        'name': 'parkinsons-drawings (Zham et al., 2017)',
        'source': 'Kaggle: kmader/parkinsons-drawings',
        'drawing_types': ['spiral', 'wave'],
        'paper': 'doi:10.3389/fneur.2017.00435',
    }
}, export_path)

size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f"✅ Full model exported: {export_path} ({size_mb:.1f} MB)")

# ═══════════════════════════════════════════════════════════
# 2. TorchScript model (mobile / ONNX)
# ═══════════════════════════════════════════════════════════
try:
    model.eval()
    
    class MotorInference(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.backbone = model.backbone
            self.classifier = model.classifier
        
        def forward(self, x):
            features = self.backbone(x)
            logits = self.classifier(features)
            return logits
    
    inference_model = MotorInference(model)
    scripted = torch.jit.trace(inference_model, torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device))
    script_path = os.path.join(EXPORT_DIR, "motor_spiral_model_mobile.pt")
    scripted.save(script_path)
    size_mb_s = os.path.getsize(script_path) / (1024 * 1024)
    print(f"✅ TorchScript model: {script_path} ({size_mb_s:.1f} MB)")
except Exception as e:
    print(f"⚠️  TorchScript export failed (non-critical): {e}")

# ═══════════════════════════════════════════════════════════
# 3. Config JSON
# ═══════════════════════════════════════════════════════════
config = {
    "model_name": "MotorNet",
    "version": "1.0",
    "architecture": "efficientnet_b0",
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "class_names": CLASS_NAMES,
    "drawing_types": ["spiral", "wave"],
    "updrs_mapping": {
        "0": {"pd_risk": 0.05, "updrs_tremor": 0, "label": "Normal — No tremor"},
        "1": {"pd_risk": 0.65, "updrs_tremor": 2, "label": "PD detected — Tremor present"},
    },
    "confidence_to_updrs": [
        {"pred": "healthy", "conf_min": 0.90, "updrs": 0, "severity": "Normal"},
        {"pred": "healthy", "conf_min": 0.70, "updrs": 1, "severity": "Slight"},
        {"pred": "parkinson", "conf_min": 0.50, "updrs": 2, "severity": "Mild"},
        {"pred": "parkinson", "conf_min": 0.70, "updrs": 3, "severity": "Moderate"},
        {"pred": "parkinson", "conf_min": 0.85, "updrs": 4, "severity": "Severe"},
    ],
    "metrics": {
        "test_accuracy": round(float(test_acc), 4),
        "test_f1": round(float(test_f1), 4),
        "best_val_f1": round(float(best_val_f1), 4),
    }
}

config_path = os.path.join(EXPORT_DIR, "motor_config.json")
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Config JSON: {config_path}")

# ═══════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"📦 EXPORTED FILES:")
print(f"{'='*60}")
for f in os.listdir(EXPORT_DIR):
    fpath = os.path.join(EXPORT_DIR, f)
    fsize = os.path.getsize(fpath) / (1024 * 1024)
    print(f"   {f:45s} {fsize:8.2f} MB")
print(f"\n   Copy these to your backend:")
print(f"   motor_spiral_model.pt  → neuroverse-backend/app/ml/models/")
print(f"   motor_config.json      → neuroverse-backend/app/ml/config/")

## 1️⃣1️⃣ Backend Integration Code

Ready-to-copy Python code for `neuroverse-backend/app/ml/predictors/motor_predictor.py`.
This handles:
1. Loading the trained model
2. Predicting PD risk from spiral/wave drawings
3. Mapping to UPDRS tremor scores
4. Integrating with `fusion_service.py` motor assessment

In [ ]:
# ============================================================
# CELL 14: Backend Integration Code (copy to your backend)
# ============================================================

backend_code = '''
# ═══════════════════════════════════════════════════════════
# FILE: neuroverse-backend/app/ml/predictors/motor_predictor.py
# ═══════════════════════════════════════════════════════════
"""
Motor Drawing Predictor for NeuroVerse Backend.
Predicts PD risk from spiral/wave drawings using trained MotorNet.
Maps to UPDRS tremor scores for fusion_service.py integration.
"""
import os
import json
import torch
import torch.nn as nn
import timm
import numpy as np
from PIL import Image
from torchvision import transforms


class MotorNet(nn.Module):
    """Motor drawing classifier — must match training architecture."""
    
    def __init__(self, num_classes=2, dropout=0.5):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=False, num_classes=0, global_pool='avg'
        )
        feature_dim = self.backbone.num_features  # 1280
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 256), nn.ReLU(inplace=True), nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.6),
            nn.Linear(256, 64), nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.Linear(64, num_classes),
        )
        self.risk_head = nn.Sequential(
            nn.Linear(num_classes, 16), nn.ReLU(inplace=True),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
        risk = self.risk_head(probs)
        return logits, risk.squeeze(-1)


class MotorPredictor:
    """Motor drawing predictor for NeuroVerse backend."""
    
    # Confidence → UPDRS tremor score mapping
    UPDRS_MAP = [
        # (predicted_class, min_confidence, updrs_score, label)
        ('healthy', 0.90, 0, 'Normal'),
        ('healthy', 0.70, 1, 'Slight tremor possible'),
        ('parkinson', 0.50, 2, 'Mild tremor'),
        ('parkinson', 0.70, 3, 'Moderate tremor'),
        ('parkinson', 0.85, 4, 'Severe tremor'),
    ]
    
    def __init__(self, model_path: str = None):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        if model_path is None:
            model_path = os.path.join(
                os.path.dirname(__file__), '..', 'models', 'motor_spiral_model.pt'
            )
        
        checkpoint = torch.load(model_path, map_location=self.device)
        config = checkpoint['model_config']
        
        self.model = MotorNet(
            num_classes=config['num_classes'],
            dropout=config['dropout']
        ).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        self.img_size = config['img_size']
        norm = checkpoint['normalization']
        self.transform = transforms.Compose([
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=norm['mean'], std=norm['std']),
        ])
        
        self.class_names = checkpoint['class_mapping']['class_names']
        self.clinical = checkpoint['clinical_thresholds']
        print(f"✅ Motor model loaded ({self.device})")
    
    def predict(self, image_path: str, drawing_type: str = 'spiral') -> dict:
        """
        Predict PD risk from a spiral/wave drawing.
        
        Returns dict with:
            - predicted_class: 'healthy' or 'parkinson'
            - confidence: float [0, 1]
            - pd_risk: float [0, 1]
            - updrs_tremor: int 0-4 (UPDRS 3.15-3.18)
            - severity_label: str
            - spiral_tremor: float [0, 1] (for fusion_service.py)
        """
        img = Image.open(image_path).convert('RGB')
        tensor = self.transform(img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            logits, risk = self.model(tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred_idx = int(logits.argmax(dim=1).item())
            confidence = float(probs[pred_idx])
        
        pred_class = 'healthy' if pred_idx == 0 else 'parkinson'
        
        # Map to UPDRS tremor score
        updrs_tremor = 0
        severity_label = 'Normal'
        
        if pred_class == 'parkinson':
            if confidence >= 0.85:
                updrs_tremor, severity_label = 4, 'Severe tremor'
            elif confidence >= 0.70:
                updrs_tremor, severity_label = 3, 'Moderate tremor'
            else:
                updrs_tremor, severity_label = 2, 'Mild tremor'
        else:
            if confidence >= 0.90:
                updrs_tremor, severity_label = 0, 'Normal'
            else:
                updrs_tremor, severity_label = 1, 'Slight tremor possible'
        
        # Compute spiral_tremor (0-1 scale) for fusion_service.py
        # Maps to existing: spiral_tremor feature used in _assess_motor()
        spiral_tremor = probs[1]  # PD probability = tremor severity proxy
        
        pd_risk_info = self.clinical['pd_risk_by_prediction'].get(pred_class, {})
        
        return {
            'predicted_class': pred_class,
            'confidence': confidence,
            'class_probabilities': {str(i): float(p) for i, p in enumerate(probs)},
            'pd_risk': pd_risk_info.get('pd_risk', 0.0) * (confidence if pred_class == 'parkinson' else 1.0),
            'ad_risk': pd_risk_info.get('ad_risk', 0.0),
            'updrs_tremor': updrs_tremor,
            'severity_label': severity_label,
            'drawing_type': drawing_type,
            # Features for fusion_service.py _assess_motor():
            'spiral_tremor': float(spiral_tremor),
            'tremor_frequency': 5.0 if pred_class == 'parkinson' and confidence > 0.70 else 0.0,
            'is_pd': pred_class == 'parkinson',
        }
    
    def predict_from_bytes(self, image_bytes: bytes, drawing_type: str = 'spiral') -> dict:
        """Predict from raw image bytes (for API endpoint)."""
        import io, tempfile
        img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
            img.save(f.name)
            result = self.predict(f.name, drawing_type)
            os.unlink(f.name)
        return result


# ═══════════════════════════════════════════════════════════
# Usage in FastAPI:
# ═══════════════════════════════════════════════════════════
#
# predictor = MotorPredictor()
#
# @router.post("/predict/spiral-drawing")
# async def predict_spiral(file: UploadFile, drawing_type: str = "spiral"):
#     image_bytes = await file.read()
#     result = predictor.predict_from_bytes(image_bytes, drawing_type)
#     return result
#
# # The result feeds directly into fusion_service.py:
# # features = {
# #     "spiral_tremor": result["spiral_tremor"],      # → UPDRS 3.15-3.18
# #     "tremor_frequency": result["tremor_frequency"], # → PD 4-6 Hz check
# #     "spiral_duration": 10.0,                        # from app timer
# # }
'''

print(backend_code)
print("\n" + "=" * 60)
print("📋 Copy the code above to:")
print("   neuroverse-backend/app/ml/predictors/motor_predictor.py")
print("=" * 60)

## 1️⃣2️⃣ Inference Demo — Predict & Visualize

In [ ]:
# ============================================================
# CELL 15: Inference Demo — Predict & Visualize
# ============================================================

def predict_and_visualize(image_path, drawing_type='spiral', show_gradcam=True):
    """Full inference pipeline with GradCAM."""
    
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = logits.argmax(dim=1).item()
        risk_val = risk.item()
    
    pred_class = CLASS_NAMES[pred_idx]
    confidence = probs[pred_idx]
    
    # UPDRS mapping
    if pred_idx == 1:  # PD
        if confidence >= 0.85:
            updrs, severity = 4, 'Severe'
        elif confidence >= 0.70:
            updrs, severity = 3, 'Moderate'
        else:
            updrs, severity = 2, 'Mild'
    else:  # Healthy
        if confidence >= 0.90:
            updrs, severity = 0, 'Normal'
        else:
            updrs, severity = 1, 'Slight'
    
    if show_gradcam:
        targets = [ClassifierOutputTarget(pred_idx)]
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
        img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
        img_np = np.array(img_resized).astype(np.float32) / 255.0
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
        
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        
        # Original
        axes[0].imshow(img_pil)
        axes[0].set_title(f'Original ({drawing_type})', fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        # GradCAM
        axes[1].imshow(cam_image)
        axes[1].set_title('GradCAM — Tremor Regions', fontsize=12, fontweight='bold')
        axes[1].axis('off')
        
        # Probability bars
        colors = ['#4ECDC4', '#FF6B6B']
        bars = axes[2].barh(
            list(CLASS_NAMES.values()),
            [probs[i] for i in range(NUM_CLASSES)],
            color=colors
        )
        axes[2].set_xlim(0, 1)
        axes[2].set_title('Class Probabilities', fontsize=12, fontweight='bold')
        for bar, p in zip(bars, probs):
            if p > 0.05:
                axes[2].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                           f'{p:.1%}', va='center', fontsize=11)
        
        title_color = 'green' if pred_idx == 0 else 'red'
        plt.suptitle(
            f"Prediction: {pred_class} (UPDRS {updrs} — {severity}) | PD Risk: {risk_val:.2f}",
            fontsize=14, fontweight='bold', color=title_color
        )
        plt.tight_layout()
        plt.show()
    
    return {
        'predicted_class': pred_class,
        'confidence': float(confidence),
        'pd_risk': float(risk_val),
        'updrs_tremor': updrs,
        'severity': severity,
        'drawing_type': drawing_type,
        'probabilities': {CLASS_NAMES[i]: float(probs[i]) for i in range(NUM_CLASSES)}
    }


# ═══════════════════════════════════════════════════════════
# Test on samples from each category
# ═══════════════════════════════════════════════════════════
print("🔍 Running inference tests...\n")

for drawing_type in ['spiral', 'wave']:
    for class_name in ['healthy', 'parkinson']:
        subset = df_test_orig[
            (df_test_orig['drawing_type'] == drawing_type) &
            (df_test_orig['class_name'] == class_name)
        ]
        if len(subset) == 0:
            continue
        
        sample = subset.sample(1, random_state=SEED).iloc[0]
        print(f"{'='*60}")
        print(f"🔍 {drawing_type.upper()} — True: {class_name.upper()}")
        print(f"{'='*60}")
        result = predict_and_visualize(sample['path'], drawing_type)
        print(f"   Result: {result['predicted_class']} | UPDRS: {result['updrs_tremor']} | "
              f"Conf: {result['confidence']:.1%} | Risk: {result['pd_risk']:.2f}")
        print()

## 1️⃣3️⃣ K-Fold Cross-Validation (Robustness Check)

Since the dataset is small (~102 images), single train/test split results can be noisy.
5-fold cross-validation gives a more reliable estimate of true model performance.

In [ ]:
# ============================================================
# CELL 16: 5-Fold Cross-Validation
# ============================================================

from sklearn.model_selection import StratifiedKFold

RUN_KFOLD = True  # Set False to skip (takes ~5x training time)
K_FOLDS = 5
KFOLD_EPOCHS = 30  # Fewer epochs per fold

if RUN_KFOLD:
    print(f"{'='*60}")
    print(f"🔄 {K_FOLDS}-Fold Cross-Validation")
    print(f"{'='*60}")
    
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_all, df_all['label'])):
        print(f"\n── Fold {fold+1}/{K_FOLDS} ──")
        
        df_fold_train = df_all.iloc[train_idx]
        df_fold_val = df_all.iloc[val_idx]
        
        # Create datasets with oversampling
        fold_train_ds = ParkinsonsDrawingDataset(df_fold_train, transform=train_transforms, oversample_factor=OVERSAMPLE)
        fold_val_ds = ParkinsonsDrawingDataset(df_fold_val, transform=val_transforms)
        
        # Sampler
        fold_counts = df_fold_train['label'].value_counts().sort_index()
        fold_weights = 1.0 / fold_counts.values
        fold_sample_weights = np.tile(
            df_fold_train['label'].map(dict(zip(fold_counts.index, fold_weights))).values,
            OVERSAMPLE
        )
        fold_sampler = WeightedRandomSampler(fold_sample_weights, len(fold_sample_weights), replacement=True)
        
        fold_train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE, sampler=fold_sampler, num_workers=2, pin_memory=True, drop_last=True)
        fold_val_loader = DataLoader(fold_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        
        # Fresh model
        fold_model = MotorNet(num_classes=2, pretrained=True, dropout=0.5).to(device)
        
        fold_bb_params = [p for n, p in fold_model.backbone.named_parameters() if p.requires_grad]
        fold_head_params = list(fold_model.classifier.parameters()) + list(fold_model.risk_head.parameters())
        fold_optimizer = optim.AdamW([
            {'params': fold_bb_params, 'lr': LEARNING_RATE / 10},
            {'params': fold_head_params, 'lr': LEARNING_RATE},
        ], weight_decay=WEIGHT_DECAY)
        fold_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(fold_optimizer, T_0=10, T_mult=2, eta_min=1e-7)
        fold_scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
        
        fold_best_f1 = 0
        fold_patience = 0
        
        for ep in range(KFOLD_EPOCHS):
            train_one_epoch(fold_model, fold_train_loader, criterion, fold_optimizer, fold_scaler, device)
            _, val_acc, val_f1, _, _, _ = validate(fold_model, fold_val_loader, criterion, device)
            fold_scheduler.step()
            
            if val_f1 > fold_best_f1:
                fold_best_f1 = val_f1
                fold_best_acc = val_acc
                fold_patience = 0
            else:
                fold_patience += 1
            
            if fold_patience >= 10:
                break
        
        # Final evaluation
        _, fold_acc, fold_f1, fold_preds, fold_labels, fold_probs = validate(
            fold_model, fold_val_loader, criterion, device
        )
        
        try:
            fold_auc = roc_auc_score(fold_labels, fold_probs[:, 1])
        except:
            fold_auc = 0.0
        
        fold_results.append({
            'fold': fold + 1,
            'accuracy': fold_best_acc,
            'f1': fold_best_f1,
            'auc': fold_auc,
        })
        
        print(f"   Fold {fold+1}: Acc={fold_best_acc:.3f} | F1={fold_best_f1:.3f} | AUC={fold_auc:.3f}")
        
        del fold_model, fold_optimizer
        torch.cuda.empty_cache() if device.type == 'cuda' else None
    
    # Summary
    df_folds = pd.DataFrame(fold_results)
    print(f"\n{'='*60}")
    print(f"📊 {K_FOLDS}-FOLD CROSS-VALIDATION RESULTS")
    print(f"{'='*60}")
    print(f"   Accuracy: {df_folds['accuracy'].mean():.3f} ± {df_folds['accuracy'].std():.3f}")
    print(f"   F1 Score: {df_folds['f1'].mean():.3f} ± {df_folds['f1'].std():.3f}")
    print(f"   ROC-AUC:  {df_folds['auc'].mean():.3f} ± {df_folds['auc'].std():.3f}")
    print(f"\n   Per-fold:")
    for _, r in df_folds.iterrows():
        print(f"     Fold {int(r['fold'])}: Acc={r['accuracy']:.3f} F1={r['f1']:.3f} AUC={r['auc']:.3f}")
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(K_FOLDS)
    width = 0.25
    ax.bar(x - width, df_folds['accuracy'], width, label='Accuracy', color='#4ECDC4')
    ax.bar(x, df_folds['f1'], width, label='F1', color='#FF6B6B')
    ax.bar(x + width, df_folds['auc'], width, label='AUC', color='#45B7D1')
    ax.set_xlabel('Fold')
    ax.set_ylabel('Score')
    ax.set_title(f'{K_FOLDS}-Fold Cross-Validation Results', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Fold {i+1}' for i in range(K_FOLDS)])
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'motor_kfold_results.png'), dpi=150, bbox_inches='tight')
    plt.show()

else:
    print("⏭️  Skipping K-Fold cross-validation")

---

## ✅ Summary — Motor Spiral Drawing Model Complete!

### What was trained:
| Component | Details |
|-----------|---------|
| **Model** | MotorNet (EfficientNet-B0 + custom PD head) |
| **Datasets** | 5 combined sources (~800+ unique images) |
| **Sources** | Kaggle: kmader, alihussain, vasantha + GitHub: HandPD, NewHandPD |
| **Task** | Binary: Healthy vs Parkinson's Disease |
| **Drawing Types** | Spiral + Wave + Meander (all motor fluidity tasks) |
| **Augmentation** | 12 transforms with adaptive oversampling (3-10x based on dataset size) |
| **Deduplication** | MD5 content hash to remove identical images across datasets |
| **Training** | AdamW + Cosine Annealing + Label Smoothing (0.15) |
| **XAI** | GradCAM heatmaps showing tremor regions |
| **Validation** | 5-Fold cross-validation for robustness |

### Multi-dataset sources:
| Dataset | Source | Images | Paper |
|---------|--------|--------|-------|
| Parkinson's Drawings | Kaggle (kmader) | ~102 | Zham et al., 2017 |
| Parkinson Drawing Spiral+Wave | Kaggle (alihussain) | ~200+ | Community |
| Spiral Drawings PD | Kaggle (vasantha) | ~150+ | Community |
| HandPD | GitHub (biolab) | ~92 | Pereira et al., 2016 |
| NewHandPD | GitHub (biolab) | ~264 | Pereira et al., 2016 |

### Exported files:
| File | Destination | Purpose |
|------|------------|---------|
| `motor_spiral_model.pt` | `app/ml/models/` | Backend inference |
| `motor_spiral_model_mobile.pt` | — | Mobile/edge deployment |
| `motor_config.json` | `app/ml/config/` | UPDRS mapping + thresholds |

### How it fits in NeuroVerse:
```
Flutter App: User draws spiral/wave on Canvas
    ↓ PNG image + stroke data (speed, pressure, tremor)
FastAPI Backend: MotorPredictor.predict(image)
    ↓ {predicted_class: "parkinson", pd_risk: 0.72, updrs_tremor: 3}
fusion_service.py _assess_motor():
    ↓ Combines spiral_tremor + tapping_rate + tremor_frequency
    ↓ Calculates Hoehn & Yahr stage
    ↓ UPDRS motor total score
xai_service.py: GradCAM tremor heatmap
    ↓ Visual explanation for doctor dashboard
```

### Integration with fusion_service.py:
The model output maps directly to existing features:
- `spiral_tremor` → `features["spiral_tremor"]` (UPDRS 3.15-3.18)
- `tremor_frequency` → `features["tremor_frequency"]` (4-6 Hz = PD range)
- `pd_risk` → used in `_assess_motor()` return dict

### NeuroVerse PD Detection Triad:
- [x] **Speech** — DementiaBank + EWA-DB ✅
- [x] **Clock Drawing** — Roboflow CDT ✅
- [x] **Motor/Spiral** — Multi-dataset (5 sources, ~800+ images) ✅ (THIS NOTEBOOK)
- [ ] **Gait** — Walking + Balance + Turn-in-Place
- [ ] **Fusion** — Combine all module scores